# Climate-Aware Capital Allocation Under NGFS Scenarios

This file documents the code and main results used in my thesis. It contains the full workflow: NGFS data preparation, region-sector construction, structural Monte Carlo default estimation, capital allocation, loss analysis, and chart generation.

To rerun the analysis, the raw input files are the NGFS workbooks in `ngfs/`. The `output/` folder contains the saved baseline tables and charts produced by the code.


In [ ]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
from statistics import NormalDist
from typing import Dict, Iterable, List, Sequence
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np


XML_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
SCENARIO_ORDER = ("DAPS", "DiRe", "HWTP", "SWUC")
SCENARIO_STRESS_PARAMS = {
    "DAPS": {
        "tail_base": 0.55,
        "tail_scale": 0.75,
        "epl_base": 0.020,
        "epl_cap": 0.100,
        "theta_boost": 0.65,
        "regime_threshold": 1.65,
        "jump_scale": 0.55,
        "lgd_jump_scale": 0.20,
    },
    "DiRe": {
        "tail_base": 0.70,
        "tail_scale": 0.95,
        "epl_base": 0.028,
        "epl_cap": 0.130,
        "theta_boost": 0.90,
        "regime_threshold": 1.55,
        "jump_scale": 0.72,
        "lgd_jump_scale": 0.26,
    },
    "HWTP": {
        "tail_base": 0.25,
        "tail_scale": 0.55,
        "epl_base": 0.010,
        "epl_cap": 0.070,
        "theta_boost": 0.35,
        "regime_threshold": 1.85,
        "jump_scale": 0.28,
        "lgd_jump_scale": 0.12,
    },
    "SWUC": {
        "tail_base": 0.85,
        "tail_scale": 1.15,
        "epl_base": 0.024,
        "epl_cap": 0.150,
        "theta_boost": 1.10,
        "regime_threshold": 1.25,
        "jump_scale": 1.10,
        "lgd_jump_scale": 0.38,
    },
}


DEFAULT_CONFIG = {
    "n_obligors": 5000,
    "maturity_years": 5.0,
    "analysis_year": "2027",
    "reference_lambda": 0.03,
    "risk_free_rate": 0.02,
    "average_production": 1.0,
    "diversification_cap": 0.02,
    "capital_budget": 100.0,
    "p0": 0.0,
    "alpha": np.array([0.0, 0.0, 0.0], dtype=float),
    "beta": np.array([0.1, 0.5, 0.8], dtype=float),
    "climate_weights": np.array([1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0], dtype=float),
    "quadrature_upper": 30.0,
    "quadrature_points": 192,
    "monte_carlo_draws": 5_000,
    "monte_carlo_chunk_size": 500,
    "monte_carlo_time_steps": 16,
    "north_america_regions": ("Canada - CAN", "Mexico - MEX"),
    "region_scope": "global",
    "seed": 20260313,
    "eta_spread": 0.05,
    "eta_quantiles": (0.05, 0.95),
    "pd_method": "structural_monte_carlo",
    "allocation_method": "capped_softmax",
    "allocation_temperature": 0.75,
    "allocation_floor_mix": 0.005,
    "loss_distribution_draws": 25_000,
    "loss_distribution_chunk_size": 1_000,
    "loss_given_default": 0.45,
    "store_loss_distribution_rows": True,
    "store_obligor_rows": True,
    "sensitivity_rate_grid": [0.0, 0.01, 0.02, 0.03, 0.04, 0.05],
    "sensitivity_loss_distribution_draws": 5_000,
    "sensitivity_monte_carlo_draws": 1_000,
    "tail_quantile": 0.90,
    "hazard_tail_blend": 0.60,
    "region_tail_blend": 0.35,
    "physical_risk_scale": 1.0,
    "pd_path_vol_scale": 0.18,
    "pd_path_barrier_scale": 0.08,
    "pd_path_regime_scale": 0.10,
    "regime_power": 1.35,
    "first_passage_reference_scale": 0.50,
}


### Explanation

This first code block defines the main model configuration. It fixes the numerical environment, the scenario order, and the baseline settings used in the thesis.

The main point of the block is that the thesis is parameterized explicitly. Every later result depends on choices recorded here: portfolio size, maturity, Monte Carlo settings, allocation rule, loss assumptions, and the stress-transmission coefficients used inside the risk engine.

The imports reflect the scope of the code used in the thesis:

- `csv`, `json`, and `pathlib` handle persistent inputs and outputs;
- `math` and `statistics.NormalDist` provide deterministic analytical pieces;
- `numpy` carries the numerical workload;
- the XML and zip utilities are used because the NGFS Excel workbooks are read directly from their internal workbook structure.

The `DEFAULT_CONFIG` object is the baseline experiment specification. It records both the paper-aligned constants that were kept and the later thesis extensions, including the structural Monte Carlo settings, region-aware scenario mapping, capped allocation rule, and loss-distribution controls.


In [ ]:
def _column_index_from_ref(ref: str) -> int:
    letters = []
    for ch in ref:
        if ch.isalpha():
            letters.append(ch)
        else:
            break
    index = 0
    for ch in letters:
        index = index * 26 + (ord(ch.upper()) - ord("A") + 1)
    return max(index - 1, 0)


def _load_shared_strings(zf: ZipFile) -> List[str]:
    try:
        root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
    except KeyError:
        return []

    values: List[str] = []
    for si in root.findall(f"{{{XML_NS}}}si"):
        text_parts = [node.text or "" for node in si.iter(f"{{{XML_NS}}}t")]
        values.append("".join(text_parts))
    return values


def _cell_value(cell: ET.Element, shared_strings: Sequence[str]) -> str:
    cell_type = cell.attrib.get("t")
    if cell_type == "s":
        value = cell.find(f"{{{XML_NS}}}v")
        if value is None or value.text is None:
            return ""
        return shared_strings[int(value.text)]
    if cell_type == "inlineStr":
        return "".join(node.text or "" for node in cell.iter(f"{{{XML_NS}}}t"))
    value = cell.find(f"{{{XML_NS}}}v")
    return "" if value is None or value.text is None else value.text


def iter_xlsx_rows(path: Path) -> Iterable[List[str]]:
    with ZipFile(path) as zf:
        shared_strings = _load_shared_strings(zf)
        with zf.open("xl/worksheets/sheet1.xml") as sheet_handle:
            for _, elem in ET.iterparse(sheet_handle, events=("end",)):
                if elem.tag != f"{{{XML_NS}}}row":
                    continue

                row_values: List[str] = []
                current_index = 0
                for cell in elem.findall(f"{{{XML_NS}}}c"):
                    target_index = _column_index_from_ref(cell.attrib.get("r", "A1"))
                    while current_index < target_index:
                        row_values.append("")
                        current_index += 1
                    row_values.append(_cell_value(cell, shared_strings))
                    current_index += 1
                elem.clear()
                yield row_values


def iter_xlsx_dicts(path: Path) -> Iterable[Dict[str, str]]:
    row_iter = iter_xlsx_rows(path)
    header = next(row_iter)
    width = len(header)
    for row in row_iter:
        if len(row) < width:
            row = row + [""] * (width - len(row))
        yield dict(zip(header, row[:width]))


### Explanation

This section handles NGFS workbook ingestion. The implementation reads workbook XML directly so the transformation from source file to model-ready records stays explicit and reproducible.

An `.xlsx` file is a zip archive of XML parts. These helpers unpack the relevant sheets, resolve shared strings, and return worksheet rows in a form that later functions can aggregate.

### What goes in

- a path to an Excel workbook,
- worksheet XML stored inside the workbook,
- shared-string tables used by Excel compression.

### What comes out

- row-level worksheet records,
- dictionaries keyed by column labels,
- a stable base layer for the NGFS scenario parsing functions that follow.

The choice to avoid a high-level Excel dependency is deliberate. It keeps the data-loading logic transparent and reduces the risk that workbook parsing decisions are hidden inside a third-party library.


In [ ]:
def normalize_sector(name: str) -> str:
    return " ".join((name or "").strip().split())


def winsorize(values: np.ndarray, lower: float = 0.05, upper: float = 0.95) -> np.ndarray:
    if values.size == 0:
        return values
    lo, hi = np.quantile(values, [lower, upper])
    return np.clip(values, lo, hi)


def rescale_eta(raw_ratio: np.ndarray, risk_free_rate: float, eta_spread: float, quantiles: Sequence[float]) -> np.ndarray:
    clipped = winsorize(np.asarray(raw_ratio, dtype=float), lower=float(quantiles[0]), upper=float(quantiles[1]))
    lo = float(np.min(clipped))
    hi = float(np.max(clipped))
    if hi - lo < 1e-12:
        normalized = np.full_like(clipped, 0.5, dtype=float)
    else:
        normalized = (clipped - lo) / (hi - lo)
    return float(risk_free_rate) + float(eta_spread) * normalized


def aggregate_mean(records: Iterable[Dict[str, object]], group_keys: Sequence[str], value_key: str) -> List[Dict[str, object]]:
    buckets: Dict[tuple, List[float]] = {}
    for record in records:
        key = tuple(record[k] for k in group_keys)
        buckets.setdefault(key, []).append(float(record[value_key]))

    aggregated: List[Dict[str, object]] = []
    for key, values in buckets.items():
        row = {group_keys[i]: key[i] for i in range(len(group_keys))}
        row[value_key] = float(sum(values) / len(values))
        aggregated.append(row)
    return aggregated


def tail_blend(values: Iterable[float], quantile: float, blend: float) -> float:
    arr = np.asarray(list(values), dtype=float)
    if arr.size == 0:
        return 0.0
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 0.0
    arr = np.abs(arr)
    arr = arr[arr > 1e-12]
    if arr.size == 0:
        return 0.0
    mean_value = float(arr.mean())
    tail_value = float(np.quantile(arr, quantile))
    return (1.0 - blend) * mean_value + blend * tail_value


def normalize_positive_scores(values: Sequence[float], upper_clip: float = 2.5) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    positive = arr[arr > 1e-12]
    if positive.size == 0:
        return np.zeros_like(arr)
    center = float(np.quantile(positive, 0.5))
    tail = float(np.quantile(positive, 0.9))
    scale = max(tail - center, 1e-12)
    normalized = np.clip((arr - center) / scale, 0.0, upper_clip)
    return normalized


### Explanation

These helper functions clean and reshape the raw data before the actual model starts. They make sure names match, extreme values do not take over, and the later steps all work with inputs on a sensible scale.

### What each helper is for

- `normalize_sector(...)`: used when the code reads sector names out of the NGFS files. It cleans up spacing and formatting so the same sector does not appear under slightly different names.
- `winsorize(...)`: used when a few values are much bigger or smaller than the rest. It trims those extremes so they do not distort the next scaling step.
- `rescale_eta(...)`: used when the synthetic firms are created. It takes the raw `a/b` ratio and turns it into the bounded `eta` value that the allocation rule uses.
- `aggregate_mean(...)`: used in the CLIMACRED preparation step. It combines repeated rows into one average value for each `(region, sector, shock)` group.
- `tail_blend(...)`: used mainly in the direct-impact preparation step. It keeps severe physical-risk values from disappearing when many rows are averaged together.
- `normalize_positive_scores(...)`: used after the raw stress scores are built. It puts them onto a common scale before the model turns them into `tail_intensity`, `epl_fraction`, `lgd_stress`, and `theta_scale`.

The main idea is that the raw workbook values are not sent straight into the credit model. They are cleaned and rescaled first so the later steps work with inputs that are stable and comparable.


In [ ]:
def prepare_direct_impacts(
    path: Path,
    raw_scenarios: str | Sequence[str],
    scenario_label: str,
    regions: Sequence[str],
    year_col: str,
    cfg: Dict[str, object],
) -> Dict[str, object]:
    if isinstance(raw_scenarios, str):
        raw_scenario_set = {raw_scenarios}
    else:
        raw_scenario_set = {str(item) for item in raw_scenarios}
    params = SCENARIO_STRESS_PARAMS[scenario_label]
    tail_quantile = float(cfg["tail_quantile"])
    region_tail_blend = float(cfg["region_tail_blend"])
    hazard_tail_blend = float(cfg["hazard_tail_blend"])

    parsed: List[Dict[str, object]] = []
    for row in iter_xlsx_dicts(path):
        if row.get("Scenario") not in raw_scenario_set:
            continue
        value = float(row.get(year_col) or 0.0)
        variable_parts = (row.get("Variable") or "").split("|")
        if len(variable_parts) < 3:
            continue
        parsed.append(
            {
                "region": row.get("Region", ""),
                "shock": variable_parts[0],
                "hazard": variable_parts[1],
                "sector": normalize_sector(variable_parts[-1]),
                "value": value,
            }
        )

    if regions:
        regional = [record for record in parsed if record["region"] in regions]
        source_scope = "north_america"
        if not regional or all(abs(float(record["value"])) < 1e-12 for record in regional):
            regional = parsed
            source_scope = "global_fallback"
    else:
        regional = parsed
        source_scope = "global"

    for record in regional:
        if record["shock"] == "labour_productivity_loss":
            record["shock"] = "productivity_loss"

    grouped: Dict[tuple[str, str, str, str], List[float]] = {}
    for record in regional:
        key = (str(record["region"]), str(record["sector"]), str(record["shock"]), str(record["hazard"]))
        grouped.setdefault(key, []).append(float(record["value"]))

    hazard_aggregates: Dict[tuple[str, str, str], List[float]] = {}
    for (region, sector, shock, _hazard), values in grouped.items():
        hazard_value = tail_blend(values, quantile=tail_quantile, blend=region_tail_blend)
        hazard_aggregates.setdefault((region, sector, shock), []).append(hazard_value)

    lookup: Dict[tuple[str, str, str], float] = {}
    entity_tail_sources: Dict[tuple[str, str], List[float]] = {}
    for (region, sector, shock), values in hazard_aggregates.items():
        aggregated_value = tail_blend(values, quantile=tail_quantile, blend=hazard_tail_blend)
        lookup[(region, sector, shock)] = aggregated_value
        entity_tail_sources.setdefault((region, sector), []).extend(values)

    entities = sorted({(region, sector) for region, sector, _shock in lookup})

    sector_rows = []
    severity_scores: List[float] = []
    hazard_scores: List[float] = []
    for region, sector in entities:
        theta_1_raw = float(lookup.get((region, sector, "productivity_loss"), 0.0))
        theta_2_raw = float(lookup.get((region, sector, "production_lost"), 0.0))
        theta_3_raw = float(lookup.get((region, sector, "capital_destruction"), 0.0))
        severity_scores.append(0.30 * theta_1_raw + 0.35 * theta_2_raw + 0.35 * theta_3_raw)
        hazard_scores.append(
            tail_blend(entity_tail_sources.get((region, sector), []), quantile=tail_quantile, blend=hazard_tail_blend)
        )
        sector_rows.append(
            {
                "scenario": scenario_label,
                "region": region,
                "sector": sector,
                "theta_1_raw": theta_1_raw,
                "theta_2_raw": theta_2_raw,
                "theta_3_raw": theta_3_raw,
                "theta_1": -(theta_1_raw / 100.0),
                "theta_2": -(theta_2_raw / 100.0),
                "theta_3": -(theta_3_raw / 100.0),
                "ngfs_pd_proxy": 0.0,
                "source_scope": source_scope,
            }
        )

    severity_norm = normalize_positive_scores(severity_scores)
    hazard_norm = normalize_positive_scores(hazard_scores)
    for idx, row in enumerate(sector_rows):
        composite = 0.6 * float(severity_norm[idx]) + 0.4 * float(hazard_norm[idx])
        tail_intensity = params["tail_base"] + params["tail_scale"] * composite
        damage_share = min(
            (0.45 * row["theta_2_raw"] + 0.55 * row["theta_3_raw"]) / 100.0,
            0.30,
        )
        epl_fraction = min(
            params["epl_cap"],
            params["epl_base"] + 0.015 * composite + 0.30 * damage_share,
        )
        lgd_stress = min(
            1.5,
            0.20 + 0.45 * composite + 0.25 * min(row["theta_3_raw"] / 2.0, 1.0),
        )
        row["tail_intensity"] = round(float(tail_intensity), 6)
        row["epl_fraction"] = round(float(epl_fraction), 6)
        row["lgd_stress"] = round(float(lgd_stress), 6)
        row["theta_scale"] = round(float(1.0 + params["theta_boost"] * composite), 6)

    return {"scenario": scenario_label, "source_scope": source_scope, "sector_rows": sector_rows}



### Explanation

This is the first place where raw climate scenario data becomes model-ready risk information.

The direct-impact workbooks contain variables associated with physical-style stress, such as:

- production lost,
- capital destroyed,
- productivity loss,
- labour productivity damage.

The code reads those rows and transforms them into a common region-sector representation used by the rest of the model.

### Which files feed this function

- `ngfs/direct_impacts_daps_IIASA_2025_07_02.xlsx`
- `ngfs/direct_impacts_dire_IIASA_2025_07_02.xlsx`

### Which columns are extracted

From each row, the function uses:

- `Scenario`
- `Region`
- `Variable`
- the selected year column, which is `2027` in the baseline run

The `Variable` field is split on `|`. The function then interprets those pieces as:

- shock family,
- hazard label,
- sector name.

### What the function does step by step

1. keep only the requested scenario labels,
2. read the selected year value,
3. split the `Variable` string into shock / hazard / sector,
4. normalize sector names,
5. optionally filter to selected regions,
6. aggregate repeated entries first within hazard groups and then across hazards,
7. build region-sector stress records.

### What comes out

For each valid region-sector entity, the function creates a stress record with fields such as:

- `theta_1`, `theta_2`, `theta_3`,
- `tail_intensity`,
- `epl_fraction`,
- `lgd_stress`,
- `theta_scale`.

### Where the equations happen

The code first constructs raw values:

- `theta_1_raw` from productivity loss,
- `theta_2_raw` from production lost,
- `theta_3_raw` from capital destruction.

Then it converts them into modeling terms by dividing by `100` and applying signs:

$$
\theta_1 = -\frac{\theta_{1,raw}}{100}, \quad
\theta_2 = -\frac{\theta_{2,raw}}{100}, \quad
\theta_3 = -\frac{\theta_{3,raw}}{100}.
$$

It also builds a composite severity score and then a tail and EPL severity layer from that score.

### Why this is important

This function is where the physical scenario stops being just a spreadsheet and starts becoming a modeling object. If someone asks, “where does DAPS actually enter the model?”, this is one of the first correct answers.

### Why region-sector matters

Earlier sector-only versions of the project compressed too much information and made `DiRe` look too similar to `DAPS`. By keeping the data at the region-sector level here, the model preserves much more of the “diverging realities” story before later compression happens.


## 5. CLIMACRED Scenario Preparation (`HWTP` and `SWUC`)



In [ ]:
def prepare_climacred(
    path: Path,
    raw_scenario: str,
    scenario_label: str,
    regions: Sequence[str],
    year_col: str,
    cfg: Dict[str, object],
) -> Dict[str, object]:
    keep_prefixes = {
        "corporate_bond_spread_adjustment",
        "corporate_bond_price_rel_adjustment",
        "equity_relative_adjustment",
        "baseline_pd",
        "pd_adjustment",
    }
    params = SCENARIO_STRESS_PARAMS[scenario_label]

    parsed: List[Dict[str, object]] = []
    for row in iter_xlsx_dicts(path):
        if row.get("Scenario") != raw_scenario:
            continue
        if regions and row.get("Region") not in regions:
            continue
        variable_parts = (row.get("Variable") or "").split("|")
        if len(variable_parts) < 2 or variable_parts[0] not in keep_prefixes:
            continue
        parsed.append(
            {
                "region": row.get("Region", ""),
                "shock": variable_parts[0],
                "sector": normalize_sector(variable_parts[1]),
                "value": float(row.get(year_col) or 0.0),
            }
        )

    sector_shock = aggregate_mean(parsed, ("region", "sector", "shock"), "value")
    lookup = {(row["region"], row["sector"], row["shock"]): row["value"] for row in sector_shock}
    entities = sorted({(str(row["region"]), str(row["sector"])) for row in sector_shock})

    raw_spread = np.array([float(lookup.get((region, sector, "corporate_bond_spread_adjustment"), 0.0)) for region, sector in entities])
    raw_price = np.array([float(lookup.get((region, sector, "corporate_bond_price_rel_adjustment"), 0.0)) for region, sector in entities])
    raw_equity = np.array([float(lookup.get((region, sector, "equity_relative_adjustment"), 0.0)) for region, sector in entities])
    raw_baseline_pd = np.array([float(lookup.get((region, sector, "baseline_pd"), 0.0)) for region, sector in entities])
    raw_pd_adjustment = np.array([float(lookup.get((region, sector, "pd_adjustment"), 0.0)) for region, sector in entities])

    raw_spread = winsorize(raw_spread, lower=0.02, upper=0.98)
    raw_price = winsorize(raw_price, lower=0.02, upper=0.98)
    raw_equity = winsorize(raw_equity, lower=0.02, upper=0.98)
    raw_pd_adjustment = winsorize(raw_pd_adjustment, lower=0.02, upper=0.98)

    downside_spread = np.maximum(raw_spread, 0.0)
    downside_price = np.maximum(-raw_price, 0.0)
    downside_equity = np.maximum(-raw_equity, 0.0)
    combined_pd_proxy = np.maximum(raw_baseline_pd + raw_pd_adjustment, 0.0) / 100.0

    spread_norm = normalize_positive_scores(downside_spread)
    price_norm = normalize_positive_scores(downside_price)
    equity_norm = normalize_positive_scores(downside_equity)
    pd_norm = normalize_positive_scores(combined_pd_proxy)
    composite_tail = 0.32 * spread_norm + 0.26 * price_norm + 0.24 * equity_norm + 0.18 * pd_norm

    sector_rows = []
    for i, (region, sector) in enumerate(entities):
        tail_intensity = params["tail_base"] + params["tail_scale"] * float(composite_tail[i])
        epl_fraction = min(
            params["epl_cap"],
            params["epl_base"] + 0.010 * float(composite_tail[i]) + 0.10 * float(combined_pd_proxy[i]),
        )
        lgd_stress = min(
            1.5,
            0.15 + 0.40 * float(composite_tail[i]) + 2.0 * min(float(combined_pd_proxy[i]), 0.20),
        )
        sector_rows.append(
            {
                "scenario": scenario_label,
                "region": region,
                "sector": sector,
                "theta_1_raw": float(raw_spread[i]),
                "theta_2_raw": float(raw_price[i]),
                "theta_3_raw": float(raw_equity[i]),
                "theta_1": -(float(raw_spread[i]) / 100.0),
                "theta_2": float(raw_price[i]) / 100.0,
                "theta_3": float(raw_equity[i]) / 100.0,
                "ngfs_pd_proxy": float(combined_pd_proxy[i]),
                "tail_intensity": round(float(tail_intensity), 6),
                "epl_fraction": round(float(epl_fraction), 6),
                "lgd_stress": round(float(lgd_stress), 6),
                "theta_scale": round(float(1.0 + params["theta_boost"] * float(composite_tail[i])), 6),
                "source_scope": "north_america" if regions else "global",
            }
        )

    return {"scenario": scenario_label, "source_scope": "north_america" if regions else "global", "sector_rows": sector_rows}



### Explanation

This section does for `HWTP` and `SWUC` what the previous section did for `DAPS` and `DiRe`, but the source data is more transition- and repricing-oriented.

### Which file feeds this function

- `ngfs/CLIMACRED_IIASA_2025_07_02.xlsx`

### Which columns are extracted

The function reads:

- `Scenario`
- `Region`
- `Variable`
- the selected year column (`2027` in the baseline run)

It keeps only rows where the `Variable` prefix is one of:

- `corporate_bond_spread_adjustment`
- `corporate_bond_price_rel_adjustment`
- `equity_relative_adjustment`
- `baseline_pd`
- `pd_adjustment`

### What the function does step by step

1. keep only the requested scenario (`HWTP` or `SWUC`),
2. optionally filter to selected regions,
3. split `Variable` into shock family and sector,
4. average repeated region-sector-shock observations,
5. build raw arrays for spread, price, equity, baseline PD, and PD adjustment,
6. winsorize those arrays to reduce instability from extreme outliers,
7. normalize them and combine them into a common composite stress signal.

### What comes out

Again, the function produces the same common stress language used elsewhere in the model:

- `theta_1`, `theta_2`, `theta_3`,
- `tail_intensity`,
- `epl_fraction`,
- `lgd_stress`,
- `theta_scale`.

### Where the equations happen

The code turns raw transition variables into normalized downside measures such as:

- positive spread widening,
- negative bond-price moves,
- negative equity moves,
- combined PD proxy.

It then builds a composite tail measure:

$$
\text{composite\_tail} = 0.32 \cdot \text{spread\_norm} + 0.26 \cdot \text{price\_norm} + 0.24 \cdot \text{equity\_norm} + 0.18 \cdot \text{pd\_norm}.
$$

That composite value is then used to build `tail_intensity`, `epl_fraction`, and `lgd_stress`.

### Why this design is useful

A natural question here is why two different workbook families both end up producing the same set of stress fields. The reason is that the later finance code needs a unified interface. Once every scenario is expressed in the same stress language, the rest of the pipeline can work with all four scenarios consistently.

### Economic interpretation

This is the transition side of the model. `HWTP` and `SWUC` are less about gradual physical degradation and more about repricing, adjustment, and abrupt transition effects. That is why the raw variables differ from the direct-impact files.

## 6. Region-Sector Factor Compression



In [ ]:
def build_sector_loadings(
    scenario_inputs: Dict[str, Dict[str, object]]
) -> tuple[Dict[tuple[str, str], np.ndarray], np.ndarray, List[tuple[str, str]]]:
    common_entities = None
    for payload in scenario_inputs.values():
        entities = {(str(row["region"]), str(row["sector"])) for row in payload["sector_rows"]}
        common_entities = entities if common_entities is None else common_entities.intersection(entities)
    sector_list = sorted(common_entities or set())

    feature_rows = []
    for entity in sector_list:
        row_features: List[float] = []
        for scenario_name in scenario_inputs:
            scenario_lookup = {
                (str(r["region"]), str(r["sector"])): r for r in scenario_inputs[scenario_name]["sector_rows"]
            }
            row = scenario_lookup[entity]
            row_features.extend(
                [
                    float(row["theta_1"]),
                    float(row["theta_2"]),
                    float(row["theta_3"]),
                    float(row.get("tail_intensity", 0.0)),
                    float(row.get("epl_fraction", 0.0)),
                ]
            )
        feature_rows.append(row_features)

    feature_matrix = np.array(feature_rows, dtype=float)
    centered = feature_matrix - feature_matrix.mean(axis=0, keepdims=True)
    scales = centered.std(axis=0, keepdims=True)
    scales[scales == 0] = 1.0
    scaled = centered / scales

    _, singular_values, vh = np.linalg.svd(scaled, full_matrices=False)
    scores = scaled @ vh[:2].T
    norms = np.linalg.norm(scores, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    loadings = scores / norms
    explained = singular_values[:2] ** 2
    explained = explained / explained.sum()
    eigenvalues = 3.0 * explained

    loading_map = {sector_list[i]: loadings[i] for i in range(len(sector_list))}
    return loading_map, eigenvalues, sector_list


### Explanation

At this stage the model has a high-dimensional stress description for many shared region-sector entities. The next task is to compress that information into a tractable dependence structure for portfolio risk.

The method used here is PCA/SVD-based compression.

### What goes in

The function consumes the scenario-prepared region-sector rows from all four scenarios. For every `(region, sector)` entity that exists across the full scenario set, it stacks the following features across scenarios:

- `theta_1`
- `theta_2`
- `theta_3`
- `tail_intensity`
- `epl_fraction`

### What comes out

- two retained eigenvalues, which measure the strength of the dominant common factors;
- two factor loadings for each shared region-sector entity;
- a common support over which synthetic firms can be generated.

The thesis keeps the two-factor representation from the original modeling tradition, but the loadings are now derived from richer region-sector stress features rather than from a thinner sector-only setup.


In [ ]:
def build_portfolio(
    sectors: Sequence[tuple[str, str]], loading_map: Dict[tuple[str, str], np.ndarray], cfg: Dict[str, object]
) -> List[Dict[str, float]]:
    rng = np.random.default_rng(int(cfg["seed"]))
    chosen_indices = rng.integers(0, len(sectors), size=int(cfg["n_obligors"]))
    a_values = rng.uniform(0.0, 0.5, size=int(cfg["n_obligors"]))
    b_values = rng.uniform(1.0, 4.0, size=int(cfg["n_obligors"]))
    sigma_values = rng.uniform(0.1, 0.5, size=int(cfg["n_obligors"]))
    rho_values = rng.uniform(-0.95, 0.95, size=int(cfg["n_obligors"]))
    raw_ratios = a_values / b_values
    eta_values = rescale_eta(
        raw_ratios,
        risk_free_rate=float(cfg["risk_free_rate"]),
        eta_spread=float(cfg["eta_spread"]),
        quantiles=cfg["eta_quantiles"],
    )
    portfolio: List[Dict[str, float]] = []
    for obligor_id, entity_index in enumerate(chosen_indices, start=1):
        region, sector = sectors[int(entity_index)]
        a = float(a_values[obligor_id - 1])
        b = float(b_values[obligor_id - 1])
        sigma = float(sigma_values[obligor_id - 1])
        rho = float(rho_values[obligor_id - 1])
        u1, u2 = loading_map[(str(region), str(sector))]
        portfolio.append(
            {
                "obligor_id": float(obligor_id),
                "region": str(region),
                "sector": str(sector),
                "p0": float(cfg["p0"]),
                "a": a,
                "b": b,
                "sigma": sigma,
                "rho": rho,
                "eta_raw_ratio": float(raw_ratios[obligor_id - 1]),
                "eta": float(eta_values[obligor_id - 1]),
                "u1": float(u1),
                "u2": float(u2),
            }
        )
    return portfolio



### Explanation

This function builds the synthetic universe of firms that the portfolio model will allocate across.

### Why the portfolio is synthetic

The thesis does not have access to a proprietary real bank book with thousands of firm-level exposures. A synthetic portfolio is therefore the practical academic choice. 

### What goes in

- the common `(region, sector)` entity list,
- the factor `loading_map`,
- the configuration settings controlling portfolio size and parameter ranges,
- the `r`-dependent `eta` scaling choice.

### What the code samples

The function samples, for each synthetic firm:

- an entity index,
- `a_i ~ U[0, 0.5]`,
- `b_i ~ U[1, 4]`,
- `sigma_i ~ U[0.1, 0.5]`,
- `rho_i ~ U[-0.95, 0.95]`.

Then it computes the raw ratio

$$
\frac{a_i}{b_i}
$$

and converts that into `eta_i` through the helper scaling rule.

### What comes out

A list of synthetic obligors. Each firm receives:

- a sector,
- a region,
- `a_i`, `b_i`,
- `sigma_i`,
- `rho_i`,
- factor loadings `u1_i`, `u2_i`,
- and the attractiveness term `eta_i`.

### Why `eta` is built this way

The current implementation sets

$$
\eta_i = r + \text{eta\_spread} \cdot \widetilde{\left(\frac{a_i}{b_i}\right)},
$$

where the raw ratio is winsorized and scaled. This preserves heterogeneity without letting `eta` dominate the model. In earlier versions, `eta` could overwhelm the scenario effect or behave too arbitrarily.


## 8. Quadrature Grid And Firm Value Construction



In [ ]:
def quadrature_grid(cfg: Dict[str, object]) -> tuple[np.ndarray, np.ndarray]:
    u = np.linspace(0.0, float(cfg["quadrature_upper"]), int(cfg["quadrature_points"]))
    step = u[1] - u[0]
    w = np.full_like(u, step)
    w[0] *= 0.5
    w[-1] *= 0.5
    return u, w


def make_value_from_x(mu_theta: float, sigma_i: float, b_i: float, epl: float, cfg: Dict[str, object], quad: tuple[np.ndarray, np.ndarray]):
    u, w = quad
    decay = np.exp(-b_i * u)
    sigma_term = (sigma_i ** 2) / (2.0 * b_i) * (1.0 - np.exp(-2.0 * b_i * u))
    base_weight = float(cfg["average_production"]) * w * np.exp(-float(cfg["risk_free_rate"]) * u + mu_theta + 0.5 * sigma_term)

    def value_from_x(x: float) -> float:
        return float(np.sum(base_weight * np.exp(decay * x)) - epl)

    return value_from_x


### Explanation

This section is where the structural part of the model becomes concrete. Instead of moving straight from scenario stress to default probability, the code first builds a firm-value function and then uses that function later when it sets the default barrier.

### Why quadrature appears here

The value formula used here contains an integral over future time. That means the code needs a way to approximate the area under a curve.

The code uses trapezoidal quadrature over a discretized time grid to approximate that integral.

### What `quadrature_grid(cfg)` does

- It creates a sequence of time points `u` from `0` to `U`, where `U` is the upper integration bound set in the config.
- It creates matching weights `w` for those time points.
- The first and last weights are halved, so the code is using the trapezoidal rule.

So if the true formula is

$$
\int_0^U f(u)\,du,
$$

the code approximates it with

$$
\sum_k f(u_k) w_k.
$$

In simple terms, the code samples the curve at many time points and adds up the pieces. That is what the grid is for.

### What `make_value_from_x(...)` does

This function takes the fixed firm and scenario inputs and turns them into a function of the current latent state `x`.

The inputs are:

- the stressed drift `mu_theta`,
- firm volatility `sigma_i`,
- the mean-reversion parameter `b_i`,
- the expected physical loss term `EPL`,
- the quadrature grid `(u, w)`.

Inside the function, the code precomputes the part that depends only on time:

- `exp(-r u)` discounts future value,
- `mu_theta` shifts the firm under the climate scenario,
- the variance term adjusts for the stochastic part of the process,
- `exp(-b_i u)` makes the effect of the current state fade over time because of mean reversion.

After that, it returns a callable function `value_from_x(x)`. That returned function still depends on `x`, but everything else is already prepared.

### Core equation

$$
V_{i,s}(x) = \int_0^U \bar{p} \exp\left(
-r u + \mu^{\theta}_{i,s} + \frac{1}{2}\sigma_i^2 \frac{1-e^{-2 b_i u}}{2 b_i} + e^{-b_i u} x
\right) du - EPL_{i,s}.
$$

In code, this becomes a weighted sum over the grid. The final `- EPL` term subtracts expected physical loss from the firm's value.

### Why the function form matters

The model does not just need one number. It needs to evaluate firm value at different `x` values later on. That is why `make_value_from_x(...)` returns a function instead of a single scalar.

That returned function is then used in `compute_obligor_metrics(...)` in three important places:

- `baseline_gross_fn(obligor["p0"])`: gets the firm's current gross value before physical loss,
- `stressed_value_fn(obligor["p0"])`: gets the firm's stressed current value,
- `baseline_value_fn(x_threshold)`: evaluates the value at the threshold state so the code can define the default barrier.

So this section is not just a mathematical detail. It is the step that turns the firm's state dynamics into a value object that the default logic can actually use.



In [ ]:
def compute_obligor_metrics(
    obligor: Dict[str, float],
    scenario_sector: Dict[str, Dict[str, float]],
    cfg: Dict[str, object],
    quad: tuple[np.ndarray, np.ndarray],
) -> Dict[str, float]:
    sector_row = scenario_sector[(obligor["region"], obligor["sector"])]
    scenario_params = SCENARIO_STRESS_PARAMS[str(sector_row["scenario"])]
    thetas = np.array([sector_row["theta_1"], sector_row["theta_2"], sector_row["theta_3"]], dtype=float)
    theta_scale = float(sector_row.get("theta_scale", 1.0))

    denom = float(cfg["risk_free_rate"]) + obligor["b"] - np.array(cfg["alpha"], dtype=float)
    gamma = theta_scale * thetas / (2.0 * np.array(cfg["beta"], dtype=float) * denom)

    mu_base = obligor["a"] - obligor["b"] * obligor["p0"]
    mu_theta = mu_base + float(np.dot(np.array(cfg["climate_weights"], dtype=float), gamma))

    baseline_gross_fn = make_value_from_x(mu_base, obligor["sigma"], obligor["b"], 0.0, cfg, quad)
    value_0_gross = baseline_gross_fn(obligor["p0"])
    epl = float(sector_row["epl_fraction"]) * value_0_gross
    stressed_value_fn = make_value_from_x(mu_theta, obligor["sigma"], obligor["b"], epl, cfg, quad)
    baseline_value_fn = make_value_from_x(mu_base, obligor["sigma"], obligor["b"], epl, cfg, quad)

    sigma_idio = obligor["sigma"] * math.sqrt(max(1.0 - obligor["rho"] ** 2, 1e-8))
    decay = math.exp(-obligor["b"] * float(cfg["maturity_years"]))
    mu_x_base = decay * obligor["p0"] + mu_base
    mu_x = decay * obligor["p0"] + mu_theta

    ref_default_prob = 1.0 - math.exp(-float(cfg["reference_lambda"]) * float(cfg["maturity_years"]))
    threshold_prob = ref_default_prob
    if str(cfg.get("pd_method", "structural_monte_carlo")) == "structural_monte_carlo":
        threshold_prob *= float(cfg.get("first_passage_reference_scale", 1.0))
    threshold_prob = min(max(threshold_prob, 1e-8), 1.0 - 1e-8)
    x_threshold = mu_x_base + sigma_idio * NormalDist().inv_cdf(threshold_prob)
    default_barrier = baseline_value_fn(x_threshold)
    tail_intensity = float(sector_row.get("tail_intensity", 0.0))
    stress_loading = max(theta_scale - 1.0, 0.0) + 0.35 * max(tail_intensity, 0.0)
    if str(cfg.get("pd_method", "structural_monte_carlo")) == "structural_monte_carlo":
        x_threshold += float(cfg.get("pd_path_barrier_scale", 0.0)) * stress_loading * sigma_idio
        default_barrier = baseline_value_fn(x_threshold)
    systemic_tail_loading = (
        scenario_params["jump_scale"]
        * (0.25 + 0.75 * tail_intensity)
        * (0.35 + 0.65 * 0.5 * (abs(obligor["u1"]) + abs(obligor["u2"])))
    )

    return {
        "theta_1": float(sector_row["theta_1"]),
        "theta_2": float(sector_row["theta_2"]),
        "theta_3": float(sector_row["theta_3"]),
        "gamma_1": float(gamma[0]),
        "gamma_2": float(gamma[1]),
        "gamma_3": float(gamma[2]),
        "mu_theta": mu_theta,
        "value_0": stressed_value_fn(obligor["p0"]),
        "epl": epl,
        "x_threshold": x_threshold,
        "default_barrier": default_barrier,
        "mu_x": mu_x,
        "sigma_idio": sigma_idio,
        "tail_intensity": tail_intensity,
        "lgd_stress": float(sector_row.get("lgd_stress", 0.0)),
        "theta_scale": theta_scale,
        "stress_loading": stress_loading,
        "systemic_tail_loading": systemic_tail_loading,
        "source_scope": str(sector_row["source_scope"]),
    }



### Explanation

This is one of the most important code sections in the thesis model, because it is the bridge between scenario data and firm-level credit risk.

### What goes in

- one synthetic firm with its own `a`, `b`, `sigma`, `rho`, `u1`, and `u2`,
- one region-sector stress record with `theta_1`, `theta_2`, `theta_3`, `tail_intensity`, `epl_fraction`, `lgd_stress`, and `theta_scale`,
- global model settings such as `r` and `lambda_ref`,
- the quadrature object created in the previous section.

### What the function computes

For each firm and scenario, the code builds:

- transformed stress terms `gamma`,
- stressed drift `mu_theta`,
- expected physical loss `epl`,
- latent-state means `mu_x_base` and `mu_x`,
- idiosyncratic volatility scale `sigma_idio`,
- default threshold `x_threshold`,
- default barrier value,
- stress-loading terms used later inside Monte Carlo.

### Where the main equations happen

1. Climate transformation:

$$
\gamma_{i,s,j} = \frac{\theta_{scale,i,s} \cdot \theta_{i,s,j}}{2 \beta_j (r + b_i - \alpha_j)}.
$$

2. Baseline drift:

$$
\mu_i^{base} = a_i - b_i p_{0,i}.
$$

3. Stressed drift:

$$
\mu_{i,s}^{\theta} = \mu_i^{base} + \sum_j \omega_j \gamma_{i,s,j}.
$$

4. Reference hazard anchor:

$$
p^{ref} = 1 - e^{-\lambda_{ref} T}.
$$

5. Barrier construction from the reference probability and idiosyncratic scale.

### Why this matters economically

This section answers the key modeling question: **how does climate stress actually get inside the firm?** The answer is that the scenario changes the firm's effective drift, its physical-loss burden, and its default barrier. That is the mechanism through which climate scenarios later change PD and allocation.

### Which outputs go to the next stage

The Monte Carlo PD engine uses `mu_theta`, `x_threshold`, `sigma`, `rho`, `u1`, `u2`, `tail_intensity`, and the stress-load terms built here.


## 10. Structural First-Passage Monte Carlo PD Engine



In [ ]:
def estimate_pds_structural_mc(
    rows: List[Dict[str, float]],
    cfg: Dict[str, object],
    eigenvalues: np.ndarray,
    rng: np.random.Generator,
) -> None:
    if not rows:
        return

    n = len(rows)
    draws = int(cfg["monte_carlo_draws"])
    chunk_size = int(cfg["monte_carlo_chunk_size"])
    time_steps = int(cfg.get("monte_carlo_time_steps", 24))
    maturity = float(cfg["maturity_years"])
    dt = maturity / max(time_steps, 1)
    counts = np.zeros(n, dtype=np.int64)

    p0 = np.array([row["p0"] for row in rows], dtype=np.float64)
    mu_theta = np.array([row["mu_theta"] for row in rows], dtype=np.float64)
    x_threshold = np.array([row["x_threshold"] for row in rows], dtype=np.float64)
    sigma = np.maximum(np.array([row["sigma"] for row in rows], dtype=np.float64), 1e-8)
    b_values = np.maximum(np.array([row["b"] for row in rows], dtype=np.float64), 1e-8)
    rho_values = np.clip(np.array([row["rho"] for row in rows], dtype=np.float64), -0.995, 0.995)
    u1 = np.array([row["u1"] for row in rows], dtype=np.float64)
    u2 = np.array([row["u2"] for row in rows], dtype=np.float64)
    tail_intensity = np.array([float(row.get("tail_intensity", 0.0)) for row in rows], dtype=np.float64)
    theta_scale = np.array([float(row.get("theta_scale", 1.0)) for row in rows], dtype=np.float64)
    stress_loading = np.array([float(row.get("stress_loading", 0.0)) for row in rows], dtype=np.float64)
    scenario_name = str(rows[0].get("scenario", "DAPS"))
    scenario_params = SCENARIO_STRESS_PARAMS[scenario_name]
    systemic_raw_1 = math.sqrt(float(eigenvalues[0])) * u1
    systemic_raw_2 = math.sqrt(float(eigenvalues[1])) * u2
    systemic_norm = np.sqrt(np.square(systemic_raw_1) + np.square(systemic_raw_2))
    systemic_norm = np.where(systemic_norm > 1e-12, systemic_norm, 1.0)
    systemic_w1 = systemic_raw_1 / systemic_norm
    systemic_w2 = systemic_raw_2 / systemic_norm
    idio_scale = np.sqrt(np.maximum(1.0 - np.square(rho_values), 1e-8))
    decay = np.exp(-b_values * dt)
    long_run = np.divide(mu_theta, b_values, out=np.zeros_like(mu_theta), where=b_values > 1e-12)
    innovation_scale = sigma * np.sqrt(np.maximum((1.0 - np.exp(-2.0 * b_values * dt)) / (2.0 * b_values), 1e-12))
    vol_multiplier = 1.0 + float(cfg.get("pd_path_vol_scale", 0.0)) * stress_loading

    for start in range(0, draws, chunk_size):
        size = min(chunk_size, draws - start)
        x_state = np.repeat(p0[:, None], size, axis=1)
        defaulted = np.zeros((n, size), dtype=bool)

        for _ in range(time_steps):
            g1 = rng.standard_normal(size)
            g2 = rng.standard_normal(size)
            z = rng.standard_normal((n, size))
            regime_radius = np.sqrt(np.square(g1) + np.square(g2))
            regime_excess = np.maximum(regime_radius - float(scenario_params["regime_threshold"]), 0.0)
            regime_jump = np.power(regime_excess, float(cfg["regime_power"]))
            common_z = systemic_w1[:, None] * g1[None, :] + systemic_w2[:, None] * g2[None, :]
            shock = rho_values[:, None] * common_z + idio_scale[:, None] * z
            regime_penalty = (
                float(cfg.get("pd_path_regime_scale", 0.0))
                * float(scenario_params["jump_scale"])
                * (0.35 + 0.65 * np.maximum(theta_scale - 1.0, 0.0))[:, None]
                * (0.25 + 0.75 * tail_intensity)[:, None]
                * regime_jump[None, :]
            )
            x_state = (
                long_run[:, None]
                + (x_state - long_run[:, None]) * decay[:, None]
                + innovation_scale[:, None] * vol_multiplier[:, None] * shock
                - regime_penalty
            )
            defaulted |= x_state <= x_threshold[:, None]

        counts += np.count_nonzero(defaulted, axis=1)

    pd_values = counts / float(draws)
    for i, row in enumerate(rows):
        row["pd"] = float(pd_values[i])
        row["pd_base"] = float(pd_values[i])
        row.pop("mu_x", None)
        row.pop("sigma_idio", None)



### Explanation

This is the core default engine of the current thesis model.

### What the code is trying to estimate

For each firm and each scenario, it wants the probability that the firm's latent state will cross the default barrier **at any time before maturity**. That is more faithful to structural credit-risk logic than simply checking the state once at the end.

### What goes in

The function extracts arrays from the list of firm rows, including:

- `p0`
- `mu_theta`
- `x_threshold`
- `sigma`
- `b`
- `rho`
- `u1`, `u2`
- `tail_intensity`
- `theta_scale`
- `stress_loading`

It also reads global simulation settings:

- number of paths,
- chunk size,
- number of time steps,
- maturity,
- eigenvalues from the factor-compression stage.

### What happens inside the simulation

1. Convert the time horizon `T` into `time_steps` discrete intervals.
2. For each simulation chunk, initialize every path at the firm's starting point.
3. At each time step, draw:
   - two common shocks `g1`, `g2`,
   - one idiosyncratic shock for each firm.
4. Build a common component from the factor loadings.
5. Add idiosyncratic noise.
6. Apply regime stress if the common shocks become extreme.
7. Update the latent state.
8. Mark a default if the latent state has crossed the barrier.
9. After all time steps, estimate PD as the share of paths that defaulted.

### First-passage idea

If `X_t` is the latent firm-state path, default happens when the path first crosses the barrier:

$$
\tau_{i,s} = \inf\{t \le T : X_t \le x_{i,s}^{thr}\}.
$$

The Monte Carlo estimator is then

$$
PD_{i,s} \approx \frac{1}{N} \sum_{k=1}^N \mathbf{1}\{\tau_{i,s}^{(k)} \le T\}.
$$

### Why the number of time steps matters

The important methodological point here is why the code simulates multiple time steps rather than just one terminal draw. First-passage default is about whether the path hits the barrier **along the way**, not only where it ends up at maturity.


### Which output goes forward

The function writes `pd` back into each firm row. That `pd` is then used immediately to create the allocation score.



In [ ]:
def capped_weighted_allocation(weights: np.ndarray, budget: float, cap: float) -> np.ndarray:
    allocations = np.zeros_like(weights, dtype=float)
    active = np.ones(weights.shape[0], dtype=bool)
    remaining_budget = float(budget)
    weights = np.maximum(weights.astype(float, copy=True), 0.0)

    while remaining_budget > 1e-10 and np.any(active):
        active_idx = np.flatnonzero(active)
        active_weights = weights[active_idx]
        total_weight = float(active_weights.sum())
        if total_weight <= 0.0:
            allocations[active_idx] += remaining_budget / active_idx.size
            break

        trial = remaining_budget * active_weights / total_weight
        capped = trial >= cap - 1e-12
        if not np.any(capped):
            allocations[active_idx] += trial
            remaining_budget = 0.0
            break

        capped_idx = active_idx[capped]
        allocations[capped_idx] = cap
        remaining_budget -= cap * capped_idx.size
        active[capped_idx] = False
        weights[capped_idx] = 0.0

    return allocations


def allocate_capital(rows: List[Dict[str, float]], cfg: Dict[str, object]) -> None:
    budget = float(cfg["capital_budget"])
    cap_per_obligor = float(cfg["diversification_cap"]) * budget
    method = str(cfg.get("allocation_method", "greedy_topk"))

    if method == "greedy_topk":
        remaining = budget
        ranked = sorted(rows, key=lambda row: row["score"], reverse=True)
        for row in ranked:
            if remaining <= 0.0:
                row["capital_allocation"] = 0.0
                continue
            allocation = min(cap_per_obligor, remaining)
            row["capital_allocation"] = allocation
            remaining -= allocation
        return

    scores = np.array([float(row["score"]) for row in rows], dtype=float)
    score_std = float(scores.std())
    if score_std <= 1e-12:
        logits = np.zeros_like(scores)
    else:
        logits = (scores - float(scores.mean())) / (score_std * float(cfg["allocation_temperature"]))
    logits = np.clip(logits - float(logits.max()), -50.0, 0.0)
    score_weights = np.exp(logits)
    uniform_weights = np.full_like(score_weights, 1.0 / max(score_weights.size, 1))
    mix = float(cfg.get("allocation_floor_mix", 0.0))
    blended_weights = (1.0 - mix) * score_weights / max(float(score_weights.sum()), 1e-12) + mix * uniform_weights
    allocations = capped_weighted_allocation(blended_weights, budget, cap_per_obligor)
    for row, allocation in zip(rows, allocations):
        row["capital_allocation"] = float(allocation)


## 12. Allocation Core


### Allocation Equation

After PD is estimated, the model turns firm risk into a portfolio decision.

The score is:

$$
score_{i,s} = \eta_i (1 - PD_{i,s})
$$

This has a simple interpretation.

- If `eta` is high, the firm is more attractive economically.
- If `PD` is high, the firm is less attractive from a risk perspective.
- Multiplying them forces the model to prefer names that are both attractive and comparatively safer.

### What the code does next

The allocator first reads the score of every firm. Then it decides how much capital each firm should receive.

There are two possible methods in the code:

- `greedy_topk`: the older hard-fill rule,
- `capped_softmax`: the current smoother thesis allocator.

The current baseline uses the smoother rule. Concretely, the code:

1. standardizes the score distribution,
2. converts scores into positive exponential weights,
3. blends those weights with a small uniform floor,
4. redistributes weights until the budget is fully used without violating the cap.

### Why this matters

A simple greedy top-k rule created very blocky outputs such as “exactly 20 firms at the cap.” The current allocator is smoother and closer to the idea that many firms can receive some capital, even though better firms still receive more.


## 13. Summaries And Sanity Checks



In [ ]:
def summarize_scenario(rows: List[Dict[str, float]], scenario_name: str) -> Dict[str, object]:
    allocations = np.array([float(row["capital_allocation"]) for row in rows], dtype=float)
    total_capital = float(allocations.sum())
    weights = allocations / max(total_capital, 1e-12)
    pds = np.array([float(row["pd"]) for row in rows], dtype=float)
    etas = np.array([float(row["eta"]) for row in rows], dtype=float)
    scores = np.array([float(row["score"]) for row in rows], dtype=float)
    positive_allocations = allocations > 1e-10
    top_20_share = float(np.sort(allocations)[-20:].sum() / max(total_capital, 1e-12))
    effective_obligors = float(1.0 / np.sum(np.square(weights))) if total_capital > 0.0 else 0.0

    return {
        "scenario": scenario_name,
        "obligors": len(rows),
        "mean_mu_theta": round(float(np.mean([row["mu_theta"] for row in rows])), 6),
        "mean_pd": round(float(np.mean(pds)), 6),
        "mean_tail_intensity": round(float(np.mean([float(row.get("tail_intensity", 0.0)) for row in rows])), 6),
        "mean_epl": round(float(np.mean([float(row.get("epl", 0.0)) for row in rows])), 6),
        "allocated_obligors": int(np.count_nonzero(positive_allocations)),
        "capital_weighted_pd": round(float(np.dot(weights, pds)), 6),
        "capital_weighted_eta": round(float(np.dot(weights, etas)), 6),
        "capital_weighted_score": round(float(np.dot(weights, scores)), 6),
        "effective_obligors": round(effective_obligors, 3),
        "top_20_capital_share": round(top_20_share, 6),
        "top_allocated_sector": top_sector_by_capital(rows),
        "top_allocated_region": top_region_by_capital(rows),
        "total_capital": round(total_capital, 6),
        "source_scope": ", ".join(sorted({str(row["source_scope"]) for row in rows})),
    }


def top_sector_by_capital(rows: List[Dict[str, float]]) -> str:
    sector_totals: Dict[str, float] = {}
    for row in rows:
        sector_totals[row["sector"]] = sector_totals.get(row["sector"], 0.0) + row["capital_allocation"]
    if not sector_totals:
        return ""
    return max(sector_totals.items(), key=lambda item: item[1])[0]


def top_region_by_capital(rows: List[Dict[str, float]]) -> str:
    region_totals: Dict[str, float] = {}
    for row in rows:
        region_totals[row["region"]] = region_totals.get(row["region"], 0.0) + row["capital_allocation"]
    if not region_totals:
        return ""
    return max(region_totals.items(), key=lambda item: item[1])[0]


def summarize_sanity(rows: List[Dict[str, float]], scenario_name: str, cfg: Dict[str, object]) -> Dict[str, object]:
    allocations = np.array([float(row["capital_allocation"]) for row in rows], dtype=float)
    scores = np.array([float(row["score"]) for row in rows], dtype=float)
    pds = np.array([float(row["pd"]) for row in rows], dtype=float)
    max_cap = float(allocations.max(initial=0.0))
    budget = float(cfg["capital_budget"])
    cap_limit = float(cfg["diversification_cap"]) * budget
    total_capital = float(allocations.sum())
    weights = allocations / max(total_capital, 1e-12)
    effective_obligors = float(1.0 / np.sum(np.square(weights))) if total_capital > 0.0 else 0.0
    top_20_share = float(np.sort(allocations)[-20:].sum() / max(total_capital, 1e-12))
    score_allocation_corr = float(np.corrcoef(scores, allocations)[0, 1]) if allocations.std() > 1e-12 and scores.std() > 1e-12 else 0.0
    pd_allocation_corr = float(np.corrcoef(pds, allocations)[0, 1]) if allocations.std() > 1e-12 and pds.std() > 1e-12 else 0.0

    return {
        "scenario": scenario_name,
        "total_capital_ok": abs(total_capital - budget) < 1e-8,
        "cap_constraint_ok": max_cap <= cap_limit + 1e-8,
        "allocated_obligors": int(np.count_nonzero(allocations > 1e-10)),
        "max_allocation": round(max_cap, 6),
        "cap_limit": round(cap_limit, 6),
        "effective_obligors": round(effective_obligors, 3),
        "top_20_capital_share": round(top_20_share, 6),
        "score_allocation_corr": round(score_allocation_corr, 6),
        "pd_allocation_corr": round(pd_allocation_corr, 6),
    }


def summarize_sector_allocations(rows: List[Dict[str, float]], scenario_name: str) -> List[Dict[str, object]]:
    grouped: Dict[str, List[Dict[str, float]]] = {}
    for row in rows:
        grouped.setdefault(row["sector"], []).append(row)

    summary_rows: List[Dict[str, object]] = []
    for sector, items in grouped.items():
        total_capital = float(sum(item["capital_allocation"] for item in items))
        if total_capital <= 0.0:
            continue
        summary_rows.append(
            {
                "scenario": scenario_name,
                "sector": sector,
                "obligor_count": len(items),
                "total_capital": round(total_capital, 6),
                "avg_pd": round(float(np.mean([item["pd"] for item in items])), 6),
                "avg_mu_theta": round(float(np.mean([item["mu_theta"] for item in items])), 6),
                "avg_eta": round(float(np.mean([item["eta"] for item in items])), 6),
                "avg_score": round(float(np.mean([item["score"] for item in items])), 6),
            }
        )

    summary_rows.sort(key=lambda row: (-row["total_capital"], row["sector"]))
    return summary_rows


def summarize_region_allocations(rows: List[Dict[str, float]], scenario_name: str) -> List[Dict[str, object]]:
    grouped: Dict[str, List[Dict[str, float]]] = {}
    for row in rows:
        grouped.setdefault(row["region"], []).append(row)

    summary_rows: List[Dict[str, object]] = []
    for region, items in grouped.items():
        total_capital = float(sum(item["capital_allocation"] for item in items))
        if total_capital <= 0.0:
            continue
        summary_rows.append(
            {
                "scenario": scenario_name,
                "region": region,
                "obligor_count": len(items),
                "total_capital": round(total_capital, 6),
                "avg_pd": round(float(np.mean([item["pd"] for item in items])), 6),
                "avg_mu_theta": round(float(np.mean([item["mu_theta"] for item in items])), 6),
                "avg_eta": round(float(np.mean([item["eta"] for item in items])), 6),
                "avg_score": round(float(np.mean([item["score"] for item in items])), 6),
            }
        )

    summary_rows.sort(key=lambda row: (-row["total_capital"], row["region"]))
    return summary_rows



### Explanation

Once firm-level PDs and allocations have been computed, we need portfolio-level summaries that a reader can actually interpret.

### What goes in

- firm-level results: PD, eta, score, capital allocation, region, sector,
- scenario labels,
- allocation settings such as budget and cap.

### What comes out

The summary layer produces things like:

- mean PD,
- capital-weighted PD,
- total capital,
- effective obligor count,
- top allocated sector,
- top allocated region,
- sanity checks for budget usage and cap compliance.

### Exact quantities computed here

- `capital_weighted_pd` is the weighted average of PD using allocation weights.
- `effective_obligors` is

$$
\frac{1}{\sum_i w_i^2},
$$

which is a common concentration measure.
- `top_20_capital_share` measures how much of the total budget sits in the largest twenty positions.
- `score_allocation_corr` and `pd_allocation_corr` check whether the allocator is behaving in the expected direction.

### Why sanity checks matter

A model can produce pretty charts and still be internally inconsistent. These checks verify things such as:

- did the model actually use the whole budget,
- did any single firm exceed the cap,
- are higher scores associated with higher allocations,
- are higher PDs associated with lower allocations.


## 14. Loss Distribution Simulation



In [ ]:
def expected_shortfall(losses: np.ndarray, level: float) -> float:
    threshold = float(np.quantile(losses, level))
    tail = losses[losses >= threshold]
    if tail.size == 0:
        return threshold
    return float(np.mean(tail))


def simulate_loss_distribution(
    rows: List[Dict[str, float]],
    scenario_name: str,
    eigenvalues: np.ndarray,
    cfg: Dict[str, object],
    rng: np.random.Generator,
    store_samples: bool = True,
) -> tuple[List[Dict[str, object]], Dict[str, object]]:
    if not rows:
        return [], {
            "scenario": scenario_name,
            "mean_loss": 0.0,
            "std_loss": 0.0,
            "p50_loss": 0.0,
            "var95_loss": 0.0,
            "var99_loss": 0.0,
            "es95_loss": 0.0,
            "es99_loss": 0.0,
            "mean_loss_pct": 0.0,
            "var95_loss_pct": 0.0,
            "var99_loss_pct": 0.0,
            "es95_loss_pct": 0.0,
            "es99_loss_pct": 0.0,
        }

    draws = int(cfg["loss_distribution_draws"])
    chunk_size = int(cfg["loss_distribution_chunk_size"])
    budget = float(cfg["capital_budget"])
    lgd = float(cfg.get("loss_given_default", 1.0))
    scenario_params = SCENARIO_STRESS_PARAMS[scenario_name]
    regime_power = float(cfg["regime_power"])
    loss_factor_df = 7.0

    mu_x = np.array(
        [
            math.exp(-float(row["b"]) * float(cfg["maturity_years"])) * float(row["p0"]) + float(row["mu_theta"])
            for row in rows
        ],
        dtype=np.float64,
    )
    x_threshold = np.array([float(row["x_threshold"]) for row in rows], dtype=np.float64)
    sigma = np.maximum(np.array([float(row["sigma"]) for row in rows], dtype=np.float64), 1e-8)
    sigma_idio = sigma * np.sqrt(np.maximum(1.0 - np.square(np.array([float(row["rho"]) for row in rows], dtype=np.float64)), 1e-8))
    u1 = np.array([float(row["u1"]) for row in rows], dtype=np.float64)
    u2 = np.array([float(row["u2"]) for row in rows], dtype=np.float64)
    exposures = lgd * np.array([float(row["capital_allocation"]) for row in rows], dtype=np.float64)
    tail_intensity = np.array([float(row.get("tail_intensity", 0.0)) for row in rows], dtype=np.float64)
    lgd_stress = np.array([float(row.get("lgd_stress", 0.0)) for row in rows], dtype=np.float64)
    systemic_tail_loading = np.array([float(row.get("systemic_tail_loading", 0.0)) for row in rows], dtype=np.float64)
    exposures = exposures * (1.0 + 0.08 * lgd_stress)

    a_state_offset = ((mu_x - x_threshold) / sigma)[:, None]
    a_state_scale = (sigma_idio / sigma)[:, None]
    systemic_scale_1 = math.sqrt(float(eigenvalues[0])) * u1[:, None]
    systemic_scale_2 = math.sqrt(float(eigenvalues[1])) * u2[:, None]

    losses: List[np.ndarray] = []
    for start in range(0, draws, chunk_size):
        size = min(chunk_size, draws - start)
        scale_mix = np.sqrt(loss_factor_df / np.maximum(rng.chisquare(loss_factor_df, size=size), 1e-8))
        g1 = rng.standard_normal(size) * scale_mix
        g2 = rng.standard_normal(size) * scale_mix
        z = rng.standard_normal((len(rows), size))
        regime_radius = np.sqrt(np.square(g1) + np.square(g2))
        regime_excess = np.maximum(regime_radius - float(scenario_params["regime_threshold"]), 0.0)
        regime_jump = np.power(regime_excess, regime_power)
        a_state = a_state_offset + a_state_scale * z
        x_state = (
            systemic_scale_1 * g1[None, :]
            + systemic_scale_2 * g2[None, :]
            + systemic_tail_loading[:, None] * regime_jump[None, :]
        )
        defaults = a_state <= x_state
        cascade_trigger = np.maximum(defaults.mean(axis=0) - 0.28, 0.0)
        if np.any(cascade_trigger > 0.0):
            margin = a_state - x_state
            cascade_penalty = (
                0.55
                * scenario_params["jump_scale"]
                * (0.15 + tail_intensity)[:, None]
                * cascade_trigger[None, :]
            )
            defaults = defaults | (margin <= cascade_penalty)
        loss_multiplier = 1.0 + scenario_params["lgd_jump_scale"] * lgd_stress[:, None] * regime_jump[None, :]
        losses.append(np.sum(exposures[:, None] * loss_multiplier * defaults.astype(np.float64), axis=0))

    loss_values = np.concatenate(losses)
    loss_pct = 100.0 * loss_values / max(budget, 1e-12)
    summary = {
        "scenario": scenario_name,
        "mean_loss": round(float(np.mean(loss_values)), 6),
        "std_loss": round(float(np.std(loss_values)), 6),
        "p50_loss": round(float(np.quantile(loss_values, 0.5)), 6),
        "var95_loss": round(float(np.quantile(loss_values, 0.95)), 6),
        "var99_loss": round(float(np.quantile(loss_values, 0.99)), 6),
        "es95_loss": round(expected_shortfall(loss_values, 0.95), 6),
        "es99_loss": round(expected_shortfall(loss_values, 0.99), 6),
        "mean_loss_pct": round(float(np.mean(loss_pct)), 6),
        "var95_loss_pct": round(float(np.quantile(loss_pct, 0.95)), 6),
        "var99_loss_pct": round(float(np.quantile(loss_pct, 0.99)), 6),
        "es95_loss_pct": round(expected_shortfall(loss_pct, 0.95), 6),
        "es99_loss_pct": round(expected_shortfall(loss_pct, 0.99), 6),
    }
    if store_samples:
        sample_rows = [
            {
                "scenario": scenario_name,
                "draw_id": draw_id + 1,
                "loss": round(float(loss), 6),
                "loss_pct": round(float(loss_pct_value), 6),
            }
            for draw_id, (loss, loss_pct_value) in enumerate(zip(loss_values, loss_pct))
        ]
    else:
        sample_rows = []
    return sample_rows, summary



### Explanation

This section moves from firm-level risk to portfolio-level loss.

### What goes in

- the allocated capital for each firm,
- scenario-specific firm PD structure,
- common-factor and idiosyncratic shocks,
- LGD assumptions,
- many simulation draws.

### What arrays are built at the start

The function constructs arrays for:

- latent-state means `mu_x`,
- default thresholds `x_threshold`,
- volatility `sigma`,
- factor loadings `u1`, `u2`,
- exposure values from capital allocation,
- `tail_intensity`,
- `lgd_stress`,
- `systemic_tail_loading`.

### What the function simulates

For each chunk of draws, it:

1. samples heavy-tailed common shocks,
2. samples firm-level idiosyncratic shocks,
3. builds default indicators,
4. adds a regime-jump effect if the common shock is severe,
5. adds a cascade correction if many defaults occur together,
6. computes total portfolio loss.

### Why the common shocks are heavy-tailed

The code uses a scale-mixture construction based on a chi-square draw. That means the loss engine is not purely Gaussian. This helps the tail behave more like a crisis distribution rather than a very smooth normal-like one.

### Tail metrics computed here

The function returns both raw simulated losses and summary statistics such as:

- mean loss,
- median loss,
- `VaR95`,
- `VaR99`,
- `ES95`,
- `ES99`.

Formally,

$$
VaR_{0.95} = \inf\{\ell : P(L \le \ell) \ge 0.95\}
$$

and

$$
ES_{0.95} = E[L \mid L \ge VaR_{0.95}].
$$

### Why this matters

Average PD is not enough for a climate-risk story. Two portfolios can have similar average PD and still be very different in the tail. That is why the model computes VaR and Expected Shortfall.

## 15. Full Workflow Orchestration



In [ ]:
def run_analysis(project_root: Path | str, config: Dict[str, object] | None = None) -> Dict[str, object]:
    cfg = dict(DEFAULT_CONFIG)
    if config:
        cfg.update(config)

    project_root = Path(project_root).resolve()
    data_dir = project_root / "ngfs"
    selected_regions: Sequence[str] = ()
    if cfg.get("region_scope") == "north_america":
        selected_regions = cfg["north_america_regions"]

    scenario_inputs = {
        "DAPS": prepare_direct_impacts(
            data_dir / "direct_impacts_daps_IIASA_2025_07_02.xlsx",
            raw_scenarios=("DAPS_AFR_R", "DAPS_ASIA", "DAPS_EUR", "DAPS_NAM", "DAPS_OCE", "DAPS_SAM")
            if not selected_regions
            else "DAPS_NAM",
            scenario_label="DAPS",
            regions=selected_regions,
            year_col=str(cfg["analysis_year"]),
            cfg=cfg,
        ),
        "DiRe": prepare_direct_impacts(
            data_dir / "direct_impacts_dire_IIASA_2025_07_02.xlsx",
            raw_scenarios="DiRe",
            scenario_label="DiRe",
            regions=selected_regions,
            year_col=str(cfg["analysis_year"]),
            cfg=cfg,
        ),
        "HWTP": prepare_climacred(
            data_dir / "CLIMACRED_IIASA_2025_07_02.xlsx",
            raw_scenario="HWTP",
            scenario_label="HWTP",
            regions=selected_regions,
            year_col=str(cfg["analysis_year"]),
            cfg=cfg,
        ),
        "SWUC": prepare_climacred(
            data_dir / "CLIMACRED_IIASA_2025_07_02.xlsx",
            raw_scenario="SWUC",
            scenario_label="SWUC",
            regions=selected_regions,
            year_col=str(cfg["analysis_year"]),
            cfg=cfg,
        ),
    }

    loading_map, eigenvalues, sector_list = build_sector_loadings(scenario_inputs)
    portfolio = build_portfolio(sector_list, loading_map, cfg)
    quad = quadrature_grid(cfg)

    scenario_results: Dict[str, List[Dict[str, float]]] = {}
    scenario_summary: List[Dict[str, object]] = []
    sanity_summary: List[Dict[str, object]] = []
    top_sector_allocations: Dict[str, List[Dict[str, object]]] = {}
    top_region_allocations: Dict[str, List[Dict[str, object]]] = {}
    loss_summary: List[Dict[str, object]] = []
    loss_distribution_rows: List[Dict[str, object]] = []
    source_notes: List[str] = []

    for offset, scenario_name in enumerate(("DAPS", "DiRe", "HWTP", "SWUC"), start=1):
        scenario_sector = {
            (row["region"], row["sector"]): row
            for row in scenario_inputs[scenario_name]["sector_rows"]
            if (row["region"], row["sector"]) in sector_list
        }
        rng = np.random.default_rng(int(cfg["seed"]) + 100 * offset)
        scenario_rows: List[Dict[str, float]] = []

        for obligor in portfolio:
            metrics = compute_obligor_metrics(obligor, scenario_sector, cfg, quad)
            row = dict(obligor)
            row.update(metrics)
            row["scenario"] = scenario_name
            scenario_rows.append(row)

        estimate_pds_structural_mc(scenario_rows, cfg, eigenvalues, rng)
        for row in scenario_rows:
            row["score"] = row["eta"] * (1.0 - row["pd"])
        allocate_capital(scenario_rows, cfg)
        scenario_results[scenario_name] = scenario_rows
        scenario_summary.append(summarize_scenario(scenario_rows, scenario_name))
        sanity_summary.append(summarize_sanity(scenario_rows, scenario_name, cfg))
        top_sector_allocations[scenario_name] = summarize_sector_allocations(scenario_rows, scenario_name)
        top_region_allocations[scenario_name] = summarize_region_allocations(scenario_rows, scenario_name)
        loss_rows, loss_stats = simulate_loss_distribution(
            scenario_rows,
            scenario_name,
            eigenvalues,
            cfg,
            np.random.default_rng(int(cfg["seed"]) + 1_000 * offset),
            store_samples=bool(cfg.get("store_loss_distribution_rows", True)),
        )
        loss_distribution_rows.extend(loss_rows)
        loss_summary.append(loss_stats)
        source_notes.append(f"{scenario_name}: {scenario_inputs[scenario_name]['source_scope']}")

    flat_rows = [row for scenario_rows in scenario_results.values() for row in scenario_rows] if bool(cfg.get("store_obligor_rows", True)) else []

    return {
        "config": cfg,
        "source_notes": source_notes,
        "sector_list": sector_list,
        "eigenvalues": [float(x) for x in eigenvalues],
        "scenario_summary": scenario_summary,
        "sanity_summary": sanity_summary,
        "top_sector_allocations": top_sector_allocations,
        "top_region_allocations": top_region_allocations,
        "loss_summary": loss_summary,
        "loss_distribution_rows": loss_distribution_rows,
        "obligor_rows": flat_rows,
    }



### Explanation

`run_analysis(...)` is the master function for the thesis pipeline. It is the clearest single place to study the model end to end.

### Which datasets are loaded here

The function explicitly loads:

- `ngfs/direct_impacts_daps_IIASA_2025_07_02.xlsx`
- `ngfs/direct_impacts_dire_IIASA_2025_07_02.xlsx`
- `ngfs/CLIMACRED_IIASA_2025_07_02.xlsx`

It assigns them to scenarios as follows:

- `DAPS` -> direct impacts file
- `DiRe` -> direct impacts file
- `HWTP` -> CLIMACRED file
- `SWUC` -> CLIMACRED file

### What it does in order

1. load and prepare scenario data,
2. build the factor structure,
3. generate the synthetic portfolio,
4. compute firm-level stressed metrics,
5. estimate PDs,
6. create the allocation score,
7. allocate capital,
8. simulate losses,
9. build summary tables.

### What the function returns

A single results dictionary containing:

- `scenario_summary`
- `sanity_summary`
- `top_sector_allocations`
- `top_region_allocations`
- `loss_summary`
- `loss_distribution_rows`
- `obligor_rows`
- metadata such as eigenvalues and source notes.

### Main Function

`run_analysis(...)` is where the dependency chain becomes explicit. It shows not just what each part does, but **what has to happen before what**.

## 16. Sensitivity Analysis With Respect To `r`



In [ ]:
def run_r_sensitivity(
    project_root: Path | str,
    rate_grid: Sequence[float] | None = None,
    config: Dict[str, object] | None = None,
) -> List[Dict[str, object]]:
    cfg = dict(DEFAULT_CONFIG)
    if config:
        cfg.update(config)

    if rate_grid is None:
        rate_grid = [float(x) for x in cfg["sensitivity_rate_grid"]]

    sensitivity_rows: List[Dict[str, object]] = []
    for rate in rate_grid:
        run_cfg = dict(cfg)
        run_cfg["risk_free_rate"] = float(rate)
        run_cfg["loss_distribution_draws"] = int(cfg["sensitivity_loss_distribution_draws"])
        run_cfg["monte_carlo_draws"] = int(cfg.get("sensitivity_monte_carlo_draws", cfg["monte_carlo_draws"]))
        run_cfg["store_loss_distribution_rows"] = False
        run_cfg["store_obligor_rows"] = False
        results = run_analysis(project_root, run_cfg)

        scenario_lookup = {row["scenario"]: row for row in results["scenario_summary"]}
        loss_lookup = {row["scenario"]: row for row in results["loss_summary"]}
        sanity_lookup = {row["scenario"]: row for row in results["sanity_summary"]}

        for scenario in SCENARIO_ORDER:
            scenario_row = scenario_lookup[scenario]
            loss_row = loss_lookup[scenario]
            sanity_row = sanity_lookup[scenario]
            sensitivity_rows.append(
                {
                    "risk_free_rate": round(float(rate), 6),
                    "scenario": scenario,
                    "capital_weighted_pd": scenario_row["capital_weighted_pd"],
                    "capital_weighted_score": scenario_row["capital_weighted_score"],
                    "effective_obligors": scenario_row["effective_obligors"],
                    "top_20_capital_share": scenario_row["top_20_capital_share"],
                    "mean_loss_pct": loss_row["mean_loss_pct"],
                    "var95_loss_pct": loss_row["var95_loss_pct"],
                    "es95_loss_pct": loss_row["es95_loss_pct"],
                    "top_allocated_sector": scenario_row["top_allocated_sector"],
                    "max_allocation": sanity_row["max_allocation"],
                }
            )

    return sensitivity_rows



### Explanation

The risk-free rate enters the model in several places, so it is worth testing how sensitive the outputs are to it.

### Why `r` matters here

Changing `r` affects:

- the attractiveness term `eta`,
- the transformed climate-stress mapping,
- the value calculation,
- the threshold anchor,
- and therefore both PDs and allocation.

### What this function actually does

The function loops over a rate grid and, for each rate:

1. copies the baseline configuration,
2. overwrites `risk_free_rate`,
3. lowers the simulation burden for sensitivity runs,
4. reruns the full model,
5. extracts the key summary metrics.

### What comes out

A table of how key metrics move with `r`, including:

- capital-weighted PD,
- capital-weighted score,
- mean loss,
- effective obligor count,
- top-20 capital share,
- top sector.

### Why this matters economically

This is not just a mathematical sensitivity. It shows how a core financial assumption can change both the level of risk and the concentration pattern of the allocated portfolio.

## 17. Output Writing And Pretty-Printing



In [ ]:
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("")
        return
    fieldnames = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_outputs(results: Dict[str, object], output_dir: Path | str) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    write_csv(output_dir / "scenario_summary.csv", list(results["scenario_summary"]))
    write_csv(output_dir / "allocation_sanity.csv", list(results["sanity_summary"]))
    write_csv(output_dir / "loss_summary.csv", list(results["loss_summary"]))
    write_csv(
        output_dir / "top_sector_allocations.csv",
        [row for scenario in results["top_sector_allocations"].values() for row in scenario],
    )
    if "top_region_allocations" in results:
        write_csv(
            output_dir / "top_region_allocations.csv",
            [row for scenario in results["top_region_allocations"].values() for row in scenario],
        )
    if "r_sensitivity_rows" in results:
        write_csv(output_dir / "r_sensitivity_summary.csv", list(results["r_sensitivity_rows"]))
    write_csv(output_dir / "loss_distribution_samples.csv", list(results["loss_distribution_rows"]))
    write_csv(output_dir / "obligor_results.csv", list(results["obligor_rows"]))

    notes = {
        "source_notes": list(results["source_notes"]),
        "eigenvalues": list(results["eigenvalues"]),
        "eta_formula": "eta = r + eta_spread * minmax(winsorized(a / b))",
        "pd_method": str(results["config"]["pd_method"]),
        "monte_carlo_draws": int(results["config"]["monte_carlo_draws"]),
        "monte_carlo_time_steps": int(results["config"].get("monte_carlo_time_steps", 0)),
        "allocation_method": str(results["config"]["allocation_method"]),
        "n_obligors": int(results["config"]["n_obligors"]),
        "loss_distribution_draws": int(results["config"]["loss_distribution_draws"]),
        "loss_given_default": float(results["config"]["loss_given_default"]),
        "loss_engine": "t-factor loss simulation with scenario-specific regime jumps, LGD stress, and cascade amplification",
        "scenario_stress_design": "region-aware NGFS aggregation, nonzero physical EPL, and structural first-passage Monte Carlo PD estimation",
        "portfolio_design": "synthetic 5,000-firm sector-region portfolio built on shared region-sector entities across scenarios",
    }
    if "r_sensitivity_rows" in results:
        notes["sensitivity_rate_grid"] = [float(row["risk_free_rate"]) for row in results["r_sensitivity_rows"][::len(SCENARIO_ORDER)]]
        notes["sensitivity_loss_distribution_draws"] = int(results["config"]["sensitivity_loss_distribution_draws"])
    (output_dir / "notes.json").write_text(json.dumps(notes, indent=2), encoding="utf-8")


def format_table(rows: List[Dict[str, object]], columns: Sequence[str] | None = None, limit: int | None = None) -> str:
    if not rows:
        return "(no rows)"
    if limit is not None:
        rows = rows[:limit]
    if columns is None:
        columns = list(rows[0].keys())

    prepared = []
    widths = {column: len(str(column)) for column in columns}
    for row in rows:
        prepared_row = {}
        for column in columns:
            value = row.get(column, "")
            prepared_row[column] = str(value)
            widths[column] = max(widths[column], len(prepared_row[column]))
        prepared.append(prepared_row)

    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)
    body = [
        " | ".join(prepared_row[column].ljust(widths[column]) for column in columns)
        for prepared_row in prepared
    ]
    return "\n".join([header, separator] + body)


### Explanation

This export layer writes the main thesis outputs to disk so the later chart and comparison sections can read the same saved results.

### What it does

The function `write_outputs(...)` writes the main thesis artifacts to disk, including:

- `scenario_summary.csv`
- `allocation_sanity.csv`
- `loss_summary.csv`
- `top_sector_allocations.csv`
- `top_region_allocations.csv`
- `r_sensitivity_summary.csv`
- `loss_distribution_samples.csv`
- `obligor_results.csv`
- `notes.json`

### Why this matters

The results are not kept only in memory. They are written out in stable files so the later plotting section uses the same saved numbers.


In [ ]:
project_root = Path.cwd()
results = run_analysis(project_root)
results["r_sensitivity_rows"] = run_r_sensitivity(project_root, config=results["config"])
write_outputs(results, project_root / "output")

print("Scenario summary")
print(format_table(results["scenario_summary"]))
print()
print("Sanity summary")
print(format_table(results["sanity_summary"]))
print()
print("Loss summary")
print(format_table(results["loss_summary"]))
print()
print("Rate sensitivity summary")
print(format_table(results["r_sensitivity_rows"], limit=12))



## 19. Figure Generation Code



In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_ORDER = ["DAPS", "DiRe", "HWTP", "SWUC"]
COLORS = {
    "DAPS": "#b35c44",
    "DiRe": "#867d52",
    "HWTP": "#2f7f5f",
    "SWUC": "#455e7c",
}


def read_csv(path: Path):
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


scenario_summary = read_csv(OUTPUT_DIR / "scenario_summary.csv")
sanity_summary = read_csv(OUTPUT_DIR / "allocation_sanity.csv")
loss_summary = read_csv(OUTPUT_DIR / "loss_summary.csv")
loss_samples = read_csv(OUTPUT_DIR / "loss_distribution_samples.csv")
sensitivity_rows = read_csv(OUTPUT_DIR / "r_sensitivity_summary.csv")
sector_rows = read_csv(OUTPUT_DIR / "top_sector_allocations.csv")
region_rows = read_csv(OUTPUT_DIR / "top_region_allocations.csv")

scenario_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
sanity_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
loss_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
sensitivity_rows.sort(key=lambda row: (float(row["risk_free_rate"]), SCENARIO_ORDER.index(row["scenario"])))


def save_pd_chart():
    scenarios = [row["scenario"] for row in scenario_summary]
    mean_pd = np.array([float(row["mean_pd"]) for row in scenario_summary])
    weighted_pd = np.array([float(row["capital_weighted_pd"]) for row in scenario_summary])
    x = np.arange(len(scenarios))
    w = 0.34

    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=180)
    ax.bar(x - w / 2, mean_pd, width=w, label="Portfolio mean PD", color="#9ca3af")
    ax.bar(x + w / 2, weighted_pd, width=w, label="Capital-weighted PD", color="#1f77b4")
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_ylabel("Probability of default")
    ax.set_title("Portfolio PD vs capital-weighted PD")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(CHART_DIR / "pd_comparison.png", bbox_inches="tight")
    plt.close(fig)


def save_pd_reduction_chart():
    scenarios = [row["scenario"] for row in scenario_summary]
    mean_pd = np.array([float(row["mean_pd"]) for row in scenario_summary])
    weighted_pd = np.array([float(row["capital_weighted_pd"]) for row in scenario_summary])
    reduction = 100.0 * (mean_pd - weighted_pd) / mean_pd
    x = np.arange(len(scenarios))

    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=180)
    ax.bar(x, reduction, color=[COLORS[s] for s in scenarios], width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_ylabel("Reduction (%)")
    ax.set_title("How much safer the allocated capital is")
    ax.grid(axis="y", alpha=0.2)
    for i, value in enumerate(reduction):
        ax.text(i, value + 0.2, f"{value:.1f}%", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(CHART_DIR / "pd_reduction.png", bbox_inches="tight")
    plt.close(fig)


def save_loss_distribution_chart():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=180)
    axes = axes.flatten()

    summary_lookup = {row["scenario"]: row for row in loss_summary}
    for idx, scenario in enumerate(SCENARIO_ORDER):
        ax = axes[idx]
        losses = np.array([float(row["loss_pct"]) for row in loss_samples if row["scenario"] == scenario], dtype=float)
        stats = summary_lookup[scenario]
        ax.hist(losses, bins=60, density=True, color=COLORS[scenario], alpha=0.75)
        ax.axvline(float(stats["mean_loss_pct"]), color="black", linestyle="-", linewidth=1.4, label="Mean")
        ax.axvline(float(stats["var95_loss_pct"]), color="#8b0000", linestyle="--", linewidth=1.2, label="VaR 95%")
        ax.axvline(float(stats["es95_loss_pct"]), color="#4b0000", linestyle=":", linewidth=1.2, label="ES 95%")
        ax.set_title(scenario)
        ax.set_xlabel("Portfolio loss (% of allocated capital)")
        ax.set_ylabel("Density")
        ax.grid(axis="y", alpha=0.2)
        ax.text(
            0.98,
            0.96,
            f"Mean {float(stats['mean_loss_pct']):.1f}%\nVaR95 {float(stats['var95_loss_pct']):.1f}%\nES95 {float(stats['es95_loss_pct']):.1f}%",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
        )
        if idx == 0:
            ax.legend(frameon=False, loc="upper left")

    fig.suptitle("Portfolio loss distribution by scenario", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(CHART_DIR / "loss_distribution.png", bbox_inches="tight")
    plt.close(fig)


def save_loss_tail_chart():
    scenarios = [row["scenario"] for row in loss_summary]
    mean_loss = np.array([float(row["mean_loss_pct"]) for row in loss_summary])
    var95 = np.array([float(row["var95_loss_pct"]) for row in loss_summary])
    es95 = np.array([float(row["es95_loss_pct"]) for row in loss_summary])
    x = np.arange(len(scenarios))
    w = 0.25

    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=180)
    ax.bar(x - w, mean_loss, width=w, label="Mean loss", color="#7c8794")
    ax.bar(x, var95, width=w, label="VaR 95%", color="#1f77b4")
    ax.bar(x + w, es95, width=w, label="Expected shortfall 95%", color="#d62728")
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_ylabel("Loss (% of allocated capital)")
    ax.set_title("Loss tail metrics by scenario")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(CHART_DIR / "loss_tail_metrics.png", bbox_inches="tight")
    plt.close(fig)


def save_r_sensitivity_chart():
    if not sensitivity_rows:
        return

    rate_grid = sorted({float(row["risk_free_rate"]) for row in sensitivity_rows})
    fig, axes = plt.subplots(2, 2, figsize=(13, 10), dpi=180)
    axes = axes.flatten()

    metrics = [
        ("capital_weighted_pd", "Capital-weighted PD"),
        ("mean_loss_pct", "Mean loss (% of capital)"),
        ("effective_obligors", "Effective number of firms"),
        ("top_20_capital_share", "Top 20 capital share"),
    ]

    for ax, (metric, title) in zip(axes, metrics):
        for scenario in SCENARIO_ORDER:
            rows = [row for row in sensitivity_rows if row["scenario"] == scenario]
            x = np.array([100.0 * float(row["risk_free_rate"]) for row in rows], dtype=float)
            y = np.array([float(row[metric]) for row in rows], dtype=float)
            if metric == "top_20_capital_share":
                y = 100.0 * y
            ax.plot(x, y, marker="o", linewidth=2, color=COLORS[scenario], label=scenario)
        ax.set_title(title)
        ax.set_xlabel("Risk-free rate r (%)")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("PD")
    axes[1].set_ylabel("Loss (%)")
    axes[2].set_ylabel("Count")
    axes[3].set_ylabel("Capital share (%)")
    axes[0].legend(frameon=False, ncol=2, loc="best")

    fig.suptitle("Sensitivity study with respect to r", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(CHART_DIR / "r_sensitivity_overview.png", bbox_inches="tight")
    plt.close(fig)


def save_concentration_chart():
    scenarios = [row["scenario"] for row in sanity_summary]
    effective_obligors = np.array([float(row["effective_obligors"]) for row in sanity_summary])
    top_20_share = 100.0 * np.array([float(row["top_20_capital_share"]) for row in sanity_summary])
    x = np.arange(len(scenarios))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), dpi=180)

    axes[0].bar(x, effective_obligors, color=[COLORS[s] for s in scenarios], width=0.6)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(scenarios)
    axes[0].set_ylabel("Effective number of firms")
    axes[0].set_title("How spread out the allocation is")
    axes[0].grid(axis="y", alpha=0.2)

    axes[1].bar(x, top_20_share, color=[COLORS[s] for s in scenarios], width=0.6)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(scenarios)
    axes[1].set_ylabel("Capital share (%)")
    axes[1].set_title("Capital absorbed by top 20 firms")
    axes[1].grid(axis="y", alpha=0.2)
    for i, value in enumerate(top_20_share):
        axes[1].text(i, value + 0.2, f"{value:.1f}%", ha="center", va="bottom", fontsize=9)

    fig.tight_layout()
    fig.savefig(CHART_DIR / "allocation_concentration.png", bbox_inches="tight")
    plt.close(fig)


def save_top_allocations_chart():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=180)
    axes = axes.flatten()

    for idx, scenario in enumerate(SCENARIO_ORDER):
        ax = axes[idx]
        rows = [row for row in sector_rows if row["scenario"] == scenario][:8]
        sectors = [row["sector"] for row in rows][::-1]
        capital = [float(row["total_capital"]) for row in rows][::-1]
        ax.barh(sectors, capital, color=COLORS[scenario])
        ax.set_title(scenario)
        ax.set_xlabel("Allocated capital")
        ax.grid(axis="x", alpha=0.2)
        ax.set_xlim(0, max(capital) + 1.5)
        for y, value in enumerate(capital):
            ax.text(value + 0.1, y, f"{value:.1f}", va="center", fontsize=9)

    fig.suptitle("Top allocated sectors by scenario", fontsize=15)
    fig.tight_layout(rect=[0.08, 0, 1, 0.97])
    fig.savefig(CHART_DIR / "top_sector_allocations.png", bbox_inches="tight")
    plt.close(fig)


def save_top_regions_chart():
    if not region_rows:
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=180)
    axes = axes.flatten()

    for idx, scenario in enumerate(SCENARIO_ORDER):
        ax = axes[idx]
        rows = [row for row in region_rows if row["scenario"] == scenario][:8]
        regions = [row["region"] for row in rows][::-1]
        capital = [float(row["total_capital"]) for row in rows][::-1]
        ax.barh(regions, capital, color=COLORS[scenario])
        ax.set_title(scenario)
        ax.set_xlabel("Allocated capital")
        ax.grid(axis="x", alpha=0.2)
        ax.set_xlim(0, max(capital) + 1.0)
        for y, value in enumerate(capital):
            ax.text(value + 0.05, y, f"{value:.1f}", va="center", fontsize=9)

    fig.suptitle("Top allocated regions by scenario", fontsize=15)
    fig.tight_layout(rect=[0.12, 0, 1, 0.97])
    fig.savefig(CHART_DIR / "top_region_allocations.png", bbox_inches="tight")
    plt.close(fig)


def save_budget_share_chart():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=180)
    axes = axes.flatten()

    for idx, scenario in enumerate(SCENARIO_ORDER):
        ax = axes[idx]
        rows = [row for row in sector_rows if row["scenario"] == scenario][:10]
        sectors = [row["sector"] for row in rows][::-1]
        shares = [float(row["total_capital"]) for row in rows][::-1]
        ax.barh(sectors, shares, color=COLORS[scenario])
        ax.set_title(scenario)
        ax.set_xlabel("Share of total capital (%)")
        ax.grid(axis="x", alpha=0.2)
        ax.set_xlim(0, max(shares) + 2.0)
        for y, value in enumerate(shares):
            ax.text(value + 0.08, y, f"{value:.1f}%", va="center", fontsize=9)

    fig.suptitle("Allocation composition by scenario", fontsize=15)
    fig.tight_layout(rect=[0.08, 0, 1, 0.97])
    fig.savefig(CHART_DIR / "allocation_budget_share.png", bbox_inches="tight")
    plt.close(fig)


def save_allocation_heatmap():
    sector_totals = {}
    for row in sector_rows:
        sector_totals[row["sector"]] = sector_totals.get(row["sector"], 0.0) + float(row["total_capital"])
    top_sectors = [sector for sector, _ in sorted(sector_totals.items(), key=lambda item: (-item[1], item[0]))[:14]]

    matrix = np.zeros((len(top_sectors), len(SCENARIO_ORDER)))
    for i, sector in enumerate(top_sectors):
        for j, scenario in enumerate(SCENARIO_ORDER):
            for row in sector_rows:
                if row["scenario"] == scenario and row["sector"] == sector:
                    matrix[i, j] = float(row["total_capital"])
                    break

    fig, ax = plt.subplots(figsize=(8.5, 8), dpi=180)
    im = ax.imshow(matrix, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(np.arange(len(SCENARIO_ORDER)))
    ax.set_xticklabels(SCENARIO_ORDER)
    ax.set_yticks(np.arange(len(top_sectors)))
    ax.set_yticklabels(top_sectors)
    ax.set_title("Allocation heatmap by sector and scenario")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if value > 0:
                ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8, color="black")
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("Allocated capital")
    fig.tight_layout()
    fig.savefig(CHART_DIR / "allocation_heatmap.png", bbox_inches="tight")
    plt.close(fig)


def save_allocation_delta_heatmap():
    sector_totals = {}
    for row in sector_rows:
        sector_totals[row["sector"]] = sector_totals.get(row["sector"], 0.0) + float(row["total_capital"])
    top_sectors = [sector for sector, _ in sorted(sector_totals.items(), key=lambda item: (-item[1], item[0]))[:14]]

    matrix = np.zeros((len(top_sectors), len(SCENARIO_ORDER)))
    for i, sector in enumerate(top_sectors):
        for j, scenario in enumerate(SCENARIO_ORDER):
            for row in sector_rows:
                if row["scenario"] == scenario and row["sector"] == sector:
                    matrix[i, j] = float(row["total_capital"])
                    break

    centered = matrix - matrix.mean(axis=1, keepdims=True)
    limit = float(np.max(np.abs(centered))) if centered.size else 1.0
    if limit == 0.0:
        limit = 1.0

    fig, ax = plt.subplots(figsize=(8.5, 8), dpi=180)
    im = ax.imshow(centered, cmap="RdBu_r", aspect="auto", vmin=-limit, vmax=limit)
    ax.set_xticks(np.arange(len(SCENARIO_ORDER)))
    ax.set_xticklabels(SCENARIO_ORDER)
    ax.set_yticks(np.arange(len(top_sectors)))
    ax.set_yticklabels(top_sectors)
    ax.set_title("Allocation deviation from sector average")
    for i in range(centered.shape[0]):
        for j in range(centered.shape[1]):
            value = centered[i, j]
            if abs(value) >= 0.3:
                ax.text(j, i, f"{value:+.1f}", ha="center", va="center", fontsize=8, color="black")
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("Capital vs sector average")
    fig.tight_layout()
    fig.savefig(CHART_DIR / "allocation_delta_heatmap.png", bbox_inches="tight")
    plt.close(fig)


def write_markdown_report():
    lines = []
    lines.append("# Allocation Results")
    lines.append("")
    lines.append(
        "Current scope: global NGFS aggregation, 2027 snapshot, rescaled firm-level eta, 5,000-firm portfolio, "
        "and a nonlinear loss engine with regime-switch amplification."
    )
    lines.append("")
    lines.append("## Scenario summary")
    lines.append("")
    lines.append("| Scenario | Mean PD | Capital-weighted PD | Effective obligors | Top 20 capital share | Top allocated sector | Top allocated region |")
    lines.append("|---|---:|---:|---:|---:|---|---|")
    for row in scenario_summary:
        lines.append(
            f"| {row['scenario']} | {float(row['mean_pd']):.6f} | {float(row['capital_weighted_pd']):.6f} | "
            f"{float(row['effective_obligors']):.1f} | {100.0 * float(row['top_20_capital_share']):.1f}% | "
            f"{row['top_allocated_sector']} | {row.get('top_allocated_region', '')} |"
        )

    lines.append("")
    lines.append("## Allocation sanity")
    lines.append("")
    lines.append("| Scenario | Budget OK | Cap OK | Score/allocation corr | PD/allocation corr |")
    lines.append("|---|---|---|---:|---:|")
    for row in sanity_summary:
        lines.append(
            f"| {row['scenario']} | {row['total_capital_ok']} | {row['cap_constraint_ok']} | "
            f"{float(row['score_allocation_corr']):.3f} | {float(row['pd_allocation_corr']):.3f} |"
        )

    lines.append("")
    lines.append("## Loss Distribution")
    lines.append("")
    lines.append("| Scenario | Mean loss | VaR 95% | VaR 99% | ES 95% | ES 99% |")
    lines.append("|---|---:|---:|---:|---:|---:|")
    for row in loss_summary:
        lines.append(
            f"| {row['scenario']} | {float(row['mean_loss_pct']):.2f}% | {float(row['var95_loss_pct']):.2f}% | "
            f"{float(row['var99_loss_pct']):.2f}% | {float(row['es95_loss_pct']):.2f}% | {float(row['es99_loss_pct']):.2f}% |"
        )

    if sensitivity_rows:
        lines.append("")
        lines.append("## Sensitivity To r")
        lines.append("")
        lines.append("| r | Scenario | Capital-weighted PD | Mean loss | Effective obligors | Top 20 share | Top sector |")
        lines.append("|---:|---|---:|---:|---:|---:|---|")
        for row in sensitivity_rows:
            lines.append(
                f"| {100.0 * float(row['risk_free_rate']):.1f}% | {row['scenario']} | {float(row['capital_weighted_pd']):.6f} | "
                f"{float(row['mean_loss_pct']):.2f}% | {float(row['effective_obligors']):.1f} | "
                f"{100.0 * float(row['top_20_capital_share']):.2f}% | {row['top_allocated_sector']} |"
            )

    for scenario in SCENARIO_ORDER:
        lines.append("")
        lines.append(f"## Top sectors: {scenario}")
        lines.append("")
        lines.append("| Sector | Total capital | Avg PD | Avg score |")
        lines.append("|---|---:|---:|---:|")
        for row in [row for row in sector_rows if row["scenario"] == scenario][:10]:
            lines.append(
                f"| {row['sector']} | {float(row['total_capital']):.3f} | {float(row['avg_pd']):.6f} | {float(row['avg_score']):.6f} |"
            )

    if region_rows:
        for scenario in SCENARIO_ORDER:
            lines.append("")
            lines.append(f"## Top regions: {scenario}")
            lines.append("")
            lines.append("| Region | Total capital | Avg PD | Avg score |")
            lines.append("|---|---:|---:|---:|")
            for row in [row for row in region_rows if row["scenario"] == scenario][:10]:
                lines.append(
                    f"| {row['region']} | {float(row['total_capital']):.3f} | {float(row['avg_pd']):.6f} | {float(row['avg_score']):.6f} |"
                )

    (OUTPUT_DIR / "results_overview.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


save_pd_chart()
save_pd_reduction_chart()
save_loss_distribution_chart()
save_loss_tail_chart()
save_r_sensitivity_chart()
save_concentration_chart()
save_top_allocations_chart()
save_top_regions_chart()
save_budget_share_chart()
save_allocation_heatmap()
save_allocation_delta_heatmap()
write_markdown_report()

print("Wrote charts to", CHART_DIR)
print("Wrote markdown report to", OUTPUT_DIR / "results_overview.md")


### Explanation

This section rebuilds the plotting layer from the saved output files. It does not recalculate the model; it reads the saved outputs and turns them into charts.

### Which files it reads

- `output/scenario_summary.csv`
- `output/allocation_sanity.csv`
- `output/loss_summary.csv`
- `output/loss_distribution_samples.csv`
- `output/r_sensitivity_summary.csv`
- `output/top_sector_allocations.csv`
- `output/top_region_allocations.csv`

### What comes out

The main saved figures for the thesis, including:

- allocation charts,
- concentration charts,
- loss-distribution charts,
- tail-metric charts,
- `r` sensitivity charts.

### Why this split is useful

The logic stays separate: first the model writes the results, then the plotting code reads those saved results and turns them into visuals. That makes it easier to trace where each chart comes from.


In [ ]:
from pathlib import Path

project_root = Path.cwd()
results = run_analysis(project_root)
output_dir = project_root / "output"
write_outputs(results, output_dir)
print(f"Wrote allocation outputs to: {output_dir}")


### Explanation

This cell runs the baseline pipeline.

### What it uses

- `run_analysis(...)` defined earlier in this file
- `write_outputs(...)` defined earlier in this file

### What it does

1. sets the current working directory as the project root,
2. calls `run_analysis(project_root)`,
3. writes the outputs to `output`,
4. prints the final output location.

### Why it is short

All of the main logic has already been defined above. This cell is only the execution step that runs the pipeline and saves the outputs.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

scenario_summary = read_csv(OUTPUT_DIR / "scenario_summary.csv")
sanity_summary = read_csv(OUTPUT_DIR / "allocation_sanity.csv")
loss_summary = read_csv(OUTPUT_DIR / "loss_summary.csv")
loss_samples = read_csv(OUTPUT_DIR / "loss_distribution_samples.csv")
sensitivity_rows = read_csv(OUTPUT_DIR / "r_sensitivity_summary.csv")
sector_rows = read_csv(OUTPUT_DIR / "top_sector_allocations.csv")
region_rows = read_csv(OUTPUT_DIR / "top_region_allocations.csv")

scenario_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
sanity_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
loss_summary.sort(key=lambda row: SCENARIO_ORDER.index(row["scenario"]))
sensitivity_rows.sort(key=lambda row: (float(row["risk_free_rate"]), SCENARIO_ORDER.index(row["scenario"])))

save_pd_chart()
save_pd_reduction_chart()
save_loss_distribution_chart()
save_loss_tail_chart()
save_r_sensitivity_chart()
save_concentration_chart()
save_top_allocations_chart()
save_top_regions_chart()
save_budget_share_chart()
save_allocation_heatmap()
save_allocation_delta_heatmap()
write_markdown_report()
print(f"Regenerated figures in: {CHART_DIR}")


## 22. Current Baseline Scenario Summary

| scenario | mean_pd | capital_weighted_pd | effective_obligor_count | top_sector | top_region | top_20_share |
| --- | --- | --- | --- | --- | --- | --- |
| DAPS | 0.624383 | 0.341547 | 806.992 | Advanced Electric Appliances | Japan - JPN | 0.101620 |
| DiRe | 0.595087 | 0.333157 | 945.215 | CSS Bio | Greece - GRC | 0.087856 |
| HWTP | 0.548806 | 0.212225 | 278.650 | Biofuels | Japan - JPN | 0.221063 |
| SWUC | 0.669119 | 0.374991 | 803.867 | Advanced Electric Appliances | Greece - GRC | 0.102221 |

## 23. Current Baseline Loss Summary

| scenario | mean_loss_pct | var_95_pct | es_95_pct | var_99_pct | es_99_pct |
| --- | --- | --- | --- | --- | --- |
| DAPS | 10.827544 | 34.572123 | 40.743524 | 44.062933 | 49.448338 |
| DiRe | 10.580613 | 31.878512 | 37.305147 | 40.521119 | 45.205064 |
| HWTP | 6.755773 | 21.001303 | 26.146392 | 29.255871 | 33.583186 |
| SWUC | 13.269103 | 43.834286 | 53.867297 | 58.123340 | 71.528625 |

## 24. Baseline `r = 2%` Sensitivity Snapshot

| scenario | risk_free_rate | capital_weighted_pd | mean_loss_pct | effective_obligor_count | top_20_share |
| --- | --- | --- | --- | --- | --- |
| DAPS | 0.02 | 0.341014 | 10.622790 | 806.286 | 0.102225 |
| DiRe | 0.02 | 0.332872 | 10.489684 | 955.973 | 0.085030 |
| HWTP | 0.02 | 0.212764 | 6.820514 | 279.987 | 0.219860 |
| SWUC | 0.02 | 0.374110 | 13.327455 | 800.252 | 0.103175 |

## 25. Interpreting The Current Results

### Main findings

- `HWTP` is the safest scenario on average.
- `SWUC` is the harshest scenario in the tail.
- `DAPS` and `DiRe` are now more differentiated than they were in the older sector-only setup, though they are still closer together than ideal.

### Why some top allocation sectors still overlap across bad scenarios

The model allocates to the best **survivors**, not to the sectors that are most strongly associated with climate stress. That means several harsh scenarios can still end up favoring a similar set of relatively resilient sectors.

## 26. Remaining Limitations

Important limitations to acknowledge:

1. the portfolio is still synthetic,
2. region weights are not calibrated to a real bank balance sheet,
3. pooled factor compression can smooth away scenario differences,
4. `DAPS` and `DiRe` are improved but still closer than ideal,
5. finite Monte Carlo precision remains a practical issue.

These are normal and acceptable limitations for this kind of thesis, as long as they are stated clearly.

## 27. Files Produced By The Workflow

Key outputs:

- `output/scenario_summary.csv`
- `output/loss_summary.csv`
- `output/r_sensitivity_summary.csv`
- `output/top_sector_allocations.csv`
- `output/top_region_allocations.csv`
- `output/charts/loss_distribution.png`
- `output/charts/loss_tail_metrics.png`
- `output/charts/top_sector_allocations.png`
- `output/charts/top_region_allocations.png`
- `output/charts/r_sensitivity_overview.png`
